# UrbanFlow AI
## Taxi Dataset Audit

### SLIIT Codefest Datathon 2026

**Objective**

This notebook performs the initial audit of the Urban Flow Analytics Taxi Dataset.

The purpose of this stage is to understand the structure, scale, data types,
coverage, completeness, and initial quality characteristics of the raw taxi
dataset before any data cleaning or machine learning is performed.

### Important principle

The raw dataset will be treated as immutable source data.

No cleaning, filtering, imputation, feature engineering, or modelling will be
performed during the initial audit.

All decisions will be documented before being applied to the data.

## 1. Audit Objectives

The initial audit will answer the following questions:

1. What files are available?
2. How large is the taxi dataset?
3. What columns are present?
4. What data types are stored in each column?
5. Which variables contain missing values?
6. Are duplicate records present?
7. What are the ranges and distributions of numerical variables?
8. What categorical values are present?
9. What is the temporal coverage of the dataset?
10. Are there immediate data-quality concerns requiring investigation?

### Deliverable

The output of this notebook will form the evidence base for the subsequent
data-quality and preprocessing decisions.

In [4]:
# ============================================================
# 1.2 Imports
# ============================================================

from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

from IPython.display import display

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Python version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Pandas version: 2.3.1
NumPy version: 1.26.4


In [6]:
# ============================================================
# 1.3 Project Configuration
# ============================================================

# Resolve the project root.
# This assumes the notebook is located inside:
# UrbanFlow_AI/notebooks/taxi/

PROJECT_ROOT = Path("../../").resolve()

TAXI_DIR = PROJECT_ROOT / "data" / "raw" / "taxi"
ZONE_DIR = PROJECT_ROOT / "data" / "raw" / "zone"

print("Project root:")
print(PROJECT_ROOT)

print("\nTaxi data directory:")
print(TAXI_DIR)

print("\nZone data directory:")
print(ZONE_DIR)

Project root:
C:\Users\arudk\Downloads\UrbanFlow_AI

Taxi data directory:
C:\Users\arudk\Downloads\UrbanFlow_AI\data\raw\taxi

Zone data directory:
C:\Users\arudk\Downloads\UrbanFlow_AI\data\raw\zone


In [8]:
# ============================================================
# 1.4 Discover Taxi Dataset Files
# ============================================================

taxi_files = sorted(TAXI_DIR.glob("*"))

print(f"Number of files found: {len(taxi_files)}")
print()

for file in taxi_files:
    if file.is_file():
        size_mb = file.stat().st_size / (1024 ** 2)

        print(
            f"{file.name:<50} "
            f"{size_mb:>10.2f} MB"
        )

Number of files found: 12

Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv          409.76 MB
Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv          469.72 MB
Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv          441.21 MB
Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv          398.64 MB
Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv          366.14 MB
Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv          435.53 MB
Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv          455.02 MB
Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv          428.29 MB
Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv          439.61 MB
Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv          379.46 MB
Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv          345.96 MB
Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv          404.99 MB


## 1.5 Inspect the Raw CSV Schema

Because the taxi dataset is very large, the complete CSV files will not be
loaded into memory during the initial audit.

At this stage, we only inspect:

- column names
- number of columns
- a very small sample of records
- automatically inferred data types from the sample
- consistency of the schema across all monthly files

This allows us to understand the structure of the dataset without performing
an expensive full-data load.

### Important

The raw CSV files are treated as immutable source data.

No cleaning, filtering, imputation, or transformation is performed here.

In [11]:
# ============================================================
# 1.5.1 Select a Representative Taxi File
# ============================================================

# Use the first monthly file for detailed initial inspection.
# Because the files are sorted alphabetically, this should correspond
# to the earliest available month.

representative_file = taxi_files[0]

print("Representative file:")
print(representative_file.name)

print("\nFile size:")
print(f"{representative_file.stat().st_size / (1024 ** 2):,.2f} MB")

Representative file:
Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv

File size:
409.76 MB


In [13]:
# ============================================================
# 1.5.2 Read a Small Sample
# ============================================================

# Read only 10 rows.
# This is intentionally tiny because the dataset is very large.

taxi_sample = pd.read_csv(
    representative_file,
    nrows=10
)

print("Sample shape:")
print(taxi_sample.shape)

print("\nColumn names:")
for i, column in enumerate(taxi_sample.columns, start=1):
    print(f"{i:>2}. {column}")

Sample shape:
(10, 20)

Column names:
 1. provider_code
 2. pickup_timestamp
 3. dropoff_timestamp
 4. rider_count
 5. distance_miles
 6. rate_class_id
 7. offline_record_flag
 8. origin_loc_id
 9. dest_loc_id
10. fare_settlement_method
11. base_fare
12. surcharge_misc
13. transit_tax
14. driver_tip_payment
15. toll_total
16. service_improvement_fee
17. charge_total
18. zone_congestion_fee
19. Airport_fee
20. congestion_relief_fee


In [15]:
# ============================================================
# 1.5.3 Display the Sample
# ============================================================

display(taxi_sample)

,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,base_fare,surcharge_misc,transit_tax,driver_tip_payment,toll_total,service_improvement_fee,charge_total,zone_congestion_fee,Airport_fee,congestion_relief_fee
0,1,2025-04-01 00:47:06,2025-04-01 01:13:25,1.0,9.50,1.0,N,138,230,1,38.7,11.00,0.5,11.65,6.94,1.0,69.79,2.5,1.75,0.75
1,2,2025-04-01 00:27:35,2025-04-01 00:38:19,2.0,3.77,1.0,N,138,92,1,17.0,6.00,0.5,4.90,0.00,1.0,31.15,0.0,1.75,0.00
2,2,2025-04-01 00:24:07,2025-04-01 00:35:12,1.0,5.41,1.0,N,132,130,1,22.6,1.00,0.5,5.37,0.00,1.0,32.22,0.0,1.75,0.00
3,1,2025-04-01 00:56:30,2025-04-01 01:00:49,2.0,0.60,1.0,N,79,4,1,6.5,4.25,0.5,2.45,0.00,1.0,14.70,2.5,0.00,0.75
4,2,2025-04-01 00:00:17,2025-04-01 00:16:19,1.0,0.43,1.0,N,161,229,2,4.4,1.00,0.5,0.00,0.00,1.0,10.15,2.5,0.00,0.75
5,7,2025-04-01 00:39:00,2025-04-01 00:39:00,1.0,0.95,1.0,N,233,164,1,5.8,0.00,0.5,0.00,0.00,1.0,11.55,2.5,0.00,0.75
6,2,2025-04-01 00:54:37,2025-04-01 01:14:10,1.0,8.94,1.0,N,138,140,1,35.9,6.00,0.5,10.57,6.94,1.0,65.16,2.5,1.75,0.00
7,2,2025-04-01 00:11:13,2025-04-01 00:28:08,2.0,8.79,1.0,N,138,116,1,35.2,6.00,0.5,5.14,6.94,1.0,56.53,0.0,1.75,0.00
8,1,2025-04-01 00:33:16,2025-04-01 00:34:46,1.0,0.50,1.0,N,239,238,1,4.4,3.50,0.5,1.85,0.00,1.0,11.25,2.5,0.00,0.00
9,2,2025-04-01 00:48:56,2025-04-01 01:18:35,1.0,16.62,2.0,N,132,162,1,70.0,0.00,0.5,16.34,6.94,1.0,99.78,2.5,1.75,0.75


In [17]:
# ============================================================
# 1.5.4 Inspect Sample Data Types
# ============================================================

print("Data types inferred from the 10-row sample:")
print()

print(taxi_sample.dtypes)

Data types inferred from the 10-row sample:

provider_code                int64
pickup_timestamp            object
dropoff_timestamp           object
rider_count                float64
distance_miles             float64
rate_class_id              float64
offline_record_flag         object
origin_loc_id                int64
dest_loc_id                  int64
fare_settlement_method       int64
base_fare                  float64
surcharge_misc             float64
transit_tax                float64
driver_tip_payment         float64
toll_total                 float64
service_improvement_fee    float64
charge_total               float64
zone_congestion_fee        float64
Airport_fee                float64
congestion_relief_fee      float64
dtype: object


In [19]:
# ============================================================
# 1.5.5 Compact Dataset Information
# ============================================================

print("Dataset information based on the sample:")
print()

taxi_sample.info()

Dataset information based on the sample:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   provider_code            10 non-null     int64  
 1   pickup_timestamp         10 non-null     object 
 2   dropoff_timestamp        10 non-null     object 
 3   rider_count              10 non-null     float64
 4   distance_miles           10 non-null     float64
 5   rate_class_id            10 non-null     float64
 6   offline_record_flag      10 non-null     object 
 7   origin_loc_id            10 non-null     int64  
 8   dest_loc_id              10 non-null     int64  
 9   fare_settlement_method   10 non-null     int64  
 10  base_fare                10 non-null     float64
 11  surcharge_misc           10 non-null     float64
 12  transit_tax              10 non-null     float64
 13  driver_tip_payment       10 non-null     

## 1.6 Verify Schema Consistency Across Monthly Files

The taxi dataset is distributed across multiple monthly CSV files.

Before combining or processing these files, we must verify that their
schemas are consistent.

For each file we will inspect:

- number of columns
- column names
- column order

No complete file will be loaded into memory.

### Why this matters

If one monthly file has a different column name, missing field, additional
field, or different column order, blindly concatenating the files could
produce incorrect results.

Schema consistency must therefore be established before creating the
consolidated dataset.

In [22]:
# ============================================================
# 1.6.1 Check Schema of Every Monthly File
# ============================================================

schema_records = []

for file in taxi_files:
    # nrows=0 reads only the CSV header.
    header = pd.read_csv(file, nrows=0)

    columns = list(header.columns)

    schema_records.append({
        "file": file.name,
        "n_columns": len(columns),
        "columns": columns
    })

schema_df = pd.DataFrame(schema_records)

print("Schema inspection completed.")
print()

display(
    schema_df[["file", "n_columns"]]
)

Schema inspection completed.



,file,n_columns
0,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,20
1,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,20
2,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,20
3,Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv,20
4,Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,20
5,Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv,20
6,Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,20
7,Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,20
8,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,20
9,Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv,20


In [24]:
# ============================================================
# 1.6.2 Compare Every File Against the First File
# ============================================================

reference_columns = schema_records[0]["columns"]

schema_results = []

for record in schema_records:

    current_columns = record["columns"]

    same_order = current_columns == reference_columns
    same_set = set(current_columns) == set(reference_columns)

    missing_columns = [
        col for col in reference_columns
        if col not in current_columns
    ]

    extra_columns = [
        col for col in current_columns
        if col not in reference_columns
    ]

    schema_results.append({
        "file": record["file"],
        "same_column_set": same_set,
        "same_column_order": same_order,
        "missing_columns": missing_columns,
        "extra_columns": extra_columns
    })

schema_check_df = pd.DataFrame(schema_results)

display(schema_check_df)

,file,same_column_set,same_column_order,missing_columns,extra_columns
0,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,True,True,[],[]
1,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,True,True,[],[]
2,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,True,True,[],[]
3,Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv,True,True,[],[]
4,Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,True,True,[],[]
5,Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv,True,True,[],[]
6,Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,True,True,[],[]
7,Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,True,True,[],[]
8,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,True,True,[],[]
9,Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv,True,True,[],[]


## 1.7 Larger Sample Inspection

The previous sections established the structure and schema consistency of the
12 monthly taxi files.

A larger sample is now inspected to understand the behaviour of the actual
data before performing the full-scale data-quality audit.

Because the complete dataset is approximately 4.9 GB, only a controlled
sample is loaded into memory at this stage.

This inspection will examine:

- missing values
- numerical distributions
- categorical/code values
- timestamp representation
- approximate memory requirements

No records are modified, deleted, imputed, or transformed permanently.

The raw CSV files remain unchanged.

In [27]:
# ============================================================
# 1.7.1 Load a Controlled Inspection Sample
# ============================================================

# Number of rows used for exploratory inspection.
# This is intentionally much smaller than the complete dataset.

SAMPLE_ROWS = 100_000

taxi_inspection_sample = pd.read_csv(
    representative_file,
    nrows=SAMPLE_ROWS
)

print("=" * 60)
print("LARGER SAMPLE INSPECTION")
print("=" * 60)

print(f"Rows loaded:    {len(taxi_inspection_sample):,}")
print(f"Columns loaded: {len(taxi_inspection_sample.columns):,}")

memory_mb = (
    taxi_inspection_sample.memory_usage(deep=True).sum()
    / (1024 ** 2)
)

print(f"Memory usage:   {memory_mb:,.2f} MB")

LARGER SAMPLE INSPECTION
Rows loaded:    100,000
Columns loaded: 20
Memory usage:   30.71 MB


## 1.7.2 Missing-Value Audit

Before performing data cleaning, we need to determine where missing values
occur in the taxi dataset.

For each column, we calculate:

- number of missing values
- percentage of missing values

This is an exploratory audit only.

No missing values will be filled, removed, or otherwise modified at this
stage.

The results will later help us decide the appropriate treatment for each
variable.

In [30]:
# ============================================================
# 1.7.2 Missing-Value Summary
# ============================================================

# Calculate the number and percentage of missing values
# for every column in the 100,000-row inspection sample.

missing_summary = pd.DataFrame({
    "column": taxi_inspection_sample.columns,
    "missing_count": taxi_inspection_sample.isna().sum().values,
    "missing_percentage": (
        taxi_inspection_sample.isna().mean().values * 100
    )
})

# Sort from highest missingness to lowest.
missing_summary = (
    missing_summary
    .sort_values(
        "missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

display(missing_summary)

,column,missing_count,missing_percentage
0,provider_code,0,0.0
1,pickup_timestamp,0,0.0
2,Airport_fee,0,0.0
3,zone_congestion_fee,0,0.0
4,charge_total,0,0.0
5,service_improvement_fee,0,0.0
6,toll_total,0,0.0
7,driver_tip_payment,0,0.0
8,transit_tax,0,0.0
9,surcharge_misc,0,0.0


## 1.7.3 Numerical Variable Audit

The numerical variables are inspected using descriptive statistics.

The purpose of this stage is to establish a baseline understanding of the
data before defining data-quality rules.

For each numerical variable we inspect:

- count
- mean
- standard deviation
- minimum
- lower quartile
- median
- upper quartile
- maximum

Extreme values are not automatically classified as errors.

A value will only be considered an anomaly later when it violates a
domain-specific or challenge-defined rule.

No records are modified or removed in this section.

In [33]:
# ============================================================
# 1.7.3 Numerical Descriptive Statistics
# ============================================================

# Identify all numerical columns automatically.
numeric_columns = (
    taxi_inspection_sample
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

print("Numerical columns detected:")
print()

for column in numeric_columns:
    print(f"- {column}")

print("\n" + "=" * 70)
print("DESCRIPTIVE STATISTICS")
print("=" * 70)

numeric_summary = (
    taxi_inspection_sample[numeric_columns]
    .describe()
    .T
)

display(numeric_summary)

Numerical columns detected:

- provider_code
- rider_count
- distance_miles
- rate_class_id
- origin_loc_id
- dest_loc_id
- fare_settlement_method
- base_fare
- surcharge_misc
- transit_tax
- driver_tip_payment
- toll_total
- service_improvement_fee
- charge_total
- zone_congestion_fee
- Airport_fee
- congestion_relief_fee

DESCRIPTIVE STATISTICS


,count,mean,std,min,25%,50%,75%,max
provider_code,100000.0,1.816940,0.619316,1.00,2.00,2.00,2.00,7.00
rider_count,100000.0,1.228210,0.666059,0.00,1.00,1.00,1.00,8.00
distance_miles,100000.0,3.275609,4.424506,0.00,0.98,1.64,3.10,75.90
rate_class_id,100000.0,2.684240,12.425599,1.00,1.00,1.00,1.00,99.00
origin_loc_id,100000.0,167.560670,62.116816,1.00,132.00,162.00,234.00,265.00
dest_loc_id,100000.0,166.912150,68.036287,1.00,132.00,163.00,234.00,265.00
fare_settlement_method,100000.0,1.215310,0.578200,1.00,1.00,1.00,1.00,4.00
base_fare,100000.0,18.297577,18.828508,-201.80,8.60,12.80,20.50,488.80
surcharge_misc,100000.0,1.781023,2.086203,-7.50,0.00,1.00,2.50,12.50
transit_tax,100000.0,0.476735,0.140486,-0.50,0.50,0.50,0.50,0.50


## 1.7.4 Code and Categorical Variable Audit

The taxi dataset contains several coded and indicator variables.

These variables are inspected against the definitions provided in the
Datathon Data Dictionary.

The purpose is to determine:

- which values actually occur
- how frequently they occur
- whether unexpected codes are present
- whether missing values are represented through special codes
- whether the observed data agrees with the supplied field definitions

No values are changed in this section.

The following variables are examined:

- provider_code
- rate_class_id
- offline_record_flag
- fare_settlement_method

In [36]:
# ============================================================
# 1.7.4 Code / Categorical Value Audit
# ============================================================

code_columns = [
    "provider_code",
    "rate_class_id",
    "offline_record_flag",
    "fare_settlement_method"
]

for column in code_columns:

    print("=" * 75)
    print(f"Column: {column}")
    print("=" * 75)

    value_counts = (
        taxi_inspection_sample[column]
        .value_counts(dropna=False)
        .sort_index()
    )

    percentage = (
        taxi_inspection_sample[column]
        .value_counts(dropna=False, normalize=True)
        .sort_index()
        * 100
    )

    category_summary = pd.DataFrame({
        "count": value_counts,
        "percentage": percentage.round(4)
    })

    display(category_summary)

    print()

Column: provider_code


,count,percentage
provider_code,,
1,22206,22.206
2,77014,77.014
7,780,0.780



Column: rate_class_id


,count,percentage
rate_class_id,,
1.0,93534,93.534
2.0,3426,3.426
3.0,289,0.289
4.0,275,0.275
5.0,840,0.840
6.0,1,0.001
99.0,1635,1.635



Column: offline_record_flag


,count,percentage
offline_record_flag,,
N,99751,99.751
Y,249,0.249



Column: fare_settlement_method


,count,percentage
fare_settlement_method,,
1,84240,84.240
2,12486,12.486
3,777,0.777
4,2497,2.497


## 1.7.5 Timestamp Audit

The pickup and drop-off timestamps are currently represented as text values
in the raw CSV.

This section investigates the timestamp fields without modifying the raw
data.

We will determine:

- whether timestamps can be parsed successfully
- the observed temporal coverage
- whether missing or malformed timestamps exist in the sample
- whether the timestamp format is consistent
- the apparent temporal range of the records

The parsed timestamps created here are temporary inspection objects only.

In [39]:
# ============================================================
# 1.7.5 Timestamp Audit
# ============================================================

timestamp_columns = [
    "pickup_timestamp",
    "dropoff_timestamp"
]

timestamp_audit = []

for column in timestamp_columns:

    # Convert to datetime only for inspection.
    # errors="coerce" converts unparseable values to NaT
    # instead of stopping the notebook.
    parsed = pd.to_datetime(
        taxi_inspection_sample[column],
        errors="coerce"
    )

    total = len(parsed)
    successful = parsed.notna().sum()
    failed = parsed.isna().sum()

    timestamp_audit.append({
        "column": column,
        "total_values": total,
        "successful_parses": successful,
        "failed_parses": failed,
        "failed_percentage": (failed / total) * 100,
        "minimum": parsed.min(),
        "maximum": parsed.max()
    })

timestamp_audit_df = pd.DataFrame(timestamp_audit)

display(timestamp_audit_df)

,column,total_values,successful_parses,failed_parses,failed_percentage,minimum,maximum
0,pickup_timestamp,100000,100000,0,0.0,2025-03-31 23:45:01,2025-04-01 22:59:56
1,dropoff_timestamp,100000,100000,0,0.0,2025-03-31 23:54:54,2025-04-02 21:55:46


In [41]:
# ============================================================
# 1.7.5.1 Inspect Timestamp Examples
# ============================================================

for column in timestamp_columns:

    print("=" * 70)
    print(f"Examples: {column}")
    print("=" * 70)

    display(
        taxi_inspection_sample[column]
        .head(10)
        .to_frame()
    )

Examples: pickup_timestamp


,pickup_timestamp
0,2025-04-01 00:47:06
1,2025-04-01 00:27:35
2,2025-04-01 00:24:07
3,2025-04-01 00:56:30
4,2025-04-01 00:00:17
5,2025-04-01 00:39:00
6,2025-04-01 00:54:37
7,2025-04-01 00:11:13
8,2025-04-01 00:33:16
9,2025-04-01 00:48:56


Examples: dropoff_timestamp


,dropoff_timestamp
0,2025-04-01 01:13:25
1,2025-04-01 00:38:19
2,2025-04-01 00:35:12
3,2025-04-01 01:00:49
4,2025-04-01 00:16:19
5,2025-04-01 00:39:00
6,2025-04-01 01:14:10
7,2025-04-01 00:28:08
8,2025-04-01 00:34:46
9,2025-04-01 01:18:35


In [43]:
# ============================================================
# 1.7.5.2 Check Timestamp String Consistency
# ============================================================

for column in timestamp_columns:

    print("=" * 70)
    print(f"Unique string lengths: {column}")
    print("=" * 70)

    string_lengths = (
        taxi_inspection_sample[column]
        .astype(str)
        .str.len()
        .value_counts()
        .sort_index()
    )

    display(
        string_lengths.to_frame(name="count")
    )

Unique string lengths: pickup_timestamp


,count
pickup_timestamp,
19,100000


Unique string lengths: dropoff_timestamp


,count
dropoff_timestamp,
19,100000


## 1.7.6 Trip Duration Audit

A valid taxi trip should have a drop-off timestamp that is equal to or later
than its pickup timestamp.

We therefore calculate the trip duration from:

    trip duration = dropoff_timestamp - pickup_timestamp

The duration is calculated temporarily for the inspection sample.

We will investigate:

- negative-duration trips
- zero-duration trips
- positive-duration trips
- minimum and maximum duration
- duration distribution

At this stage, no records are removed or modified.

### Important

A zero-duration trip is not automatically treated as invalid.

It will be reported separately because the challenge specifically requires
investigation of temporal anomalies, while the correct treatment must be
determined from the data and business context.

In [46]:
# ============================================================
# 1.7.6.1 Calculate Trip Duration
# ============================================================

# Create temporary datetime representations.
# The original sample columns are not modified.

pickup_dt = pd.to_datetime(
    taxi_inspection_sample["pickup_timestamp"],
    errors="coerce"
)

dropoff_dt = pd.to_datetime(
    taxi_inspection_sample["dropoff_timestamp"],
    errors="coerce"
)

# Calculate trip duration in seconds.
trip_duration_seconds = (
    dropoff_dt - pickup_dt
).dt.total_seconds()

print("=" * 70)
print("TRIP DURATION SUMMARY")
print("=" * 70)

print(
    f"Minimum duration: "
    f"{trip_duration_seconds.min():,.2f} seconds"
)

print(
    f"Maximum duration: "
    f"{trip_duration_seconds.max():,.2f} seconds"
)

print(
    f"Mean duration: "
    f"{trip_duration_seconds.mean():,.2f} seconds"
)

print(
    f"Median duration: "
    f"{trip_duration_seconds.median():,.2f} seconds"
)

TRIP DURATION SUMMARY
Minimum duration: 0.00 seconds
Maximum duration: 86,330.00 seconds
Mean duration: 996.96 seconds
Median duration: 742.00 seconds


In [48]:
# ============================================================
# 1.7.6.2 Classify Trip Duration Relationships
# ============================================================

negative_duration = trip_duration_seconds < 0
zero_duration = trip_duration_seconds == 0
positive_duration = trip_duration_seconds > 0

duration_summary = pd.DataFrame({
    "duration_category": [
        "Negative duration",
        "Zero duration",
        "Positive duration"
    ],
    "count": [
        negative_duration.sum(),
        zero_duration.sum(),
        positive_duration.sum()
    ]
})

duration_summary["percentage"] = (
    duration_summary["count"]
    / len(taxi_inspection_sample)
    * 100
)

display(duration_summary)

,duration_category,count,percentage
0,Negative duration,0,0.000
1,Zero duration,804,0.804
2,Positive duration,99196,99.196


In [50]:
# ============================================================
# 1.7.6.3 Inspect Potential Temporal Anomalies
# ============================================================

duration_check = taxi_inspection_sample[
    negative_duration | zero_duration
].copy()

# Add temporary duration information for investigation.
duration_check["trip_duration_seconds"] = (
    trip_duration_seconds[
        negative_duration | zero_duration
    ].values
)

print(
    f"Potential temporal records found: "
    f"{len(duration_check):,}"
)

display(duration_check.head(20))

Potential temporal records found: 804


,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,...,surcharge_misc,transit_tax,driver_tip_payment,toll_total,service_improvement_fee,charge_total,zone_congestion_fee,Airport_fee,congestion_relief_fee,trip_duration_seconds
5,7,2025-04-01 00:39:00,2025-04-01 00:39:00,1.0,0.95,1.0,N,233,164,1,...,0.0,0.5,0.00,0.00,1.0,11.55,2.5,0.00,0.75,0.0
46,7,2025-04-01 00:25:00,2025-04-01 00:25:00,1.0,2.04,1.0,N,239,230,2,...,0.0,0.5,0.00,0.00,1.0,17.85,2.5,0.00,0.75,0.0
322,7,2025-04-01 00:04:00,2025-04-01 00:04:00,1.0,0.68,1.0,N,142,163,2,...,0.0,0.5,0.00,0.00,1.0,12.25,2.5,0.00,0.75,0.0
343,7,2025-04-01 00:31:00,2025-04-01 00:31:00,1.0,9.52,1.0,N,138,48,1,...,0.0,0.5,12.05,6.94,1.0,72.29,2.5,6.75,0.75,0.0
566,7,2025-04-01 00:27:00,2025-04-01 00:27:00,1.0,8.79,1.0,N,230,127,1,...,0.0,0.5,5.00,0.00,1.0,45.95,2.5,0.00,0.75,0.0
1149,7,2025-04-01 00:15:00,2025-04-01 00:15:00,1.0,16.95,2.0,N,132,170,1,...,0.0,0.5,16.69,6.94,1.0,100.13,2.5,1.75,0.75,0.0
1150,7,2025-04-01 00:27:00,2025-04-01 00:27:00,1.0,0.66,1.0,N,161,48,1,...,0.0,0.5,2.17,0.00,1.0,13.02,2.5,0.00,0.75,0.0
2102,7,2025-04-01 01:29:00,2025-04-01 01:29:00,1.0,8.63,1.0,N,138,97,1,...,0.0,0.5,8.47,0.00,1.0,50.82,0.0,6.75,0.00,0.0
2481,7,2025-04-01 02:09:00,2025-04-01 02:09:00,1.0,9.02,1.0,N,138,230,1,...,0.0,0.5,11.35,6.94,1.0,68.09,2.5,6.75,0.75,0.0
2871,7,2025-04-01 02:45:00,2025-04-01 02:45:00,1.0,17.91,1.0,N,132,4,1,...,0.0,0.5,12.00,0.00,1.0,90.40,2.5,1.75,0.75,0.0


## 1.7.7 Zero-Duration Trip Investigation

The previous audit identified 804 zero-duration trips in the
100,000-row inspection sample.

A zero-duration trip does not necessarily mean that the entire
record is invalid. Therefore, before deciding whether these records
should be removed, we investigate their associated trip attributes.

We specifically examine:

- distance traveled
- base fare
- total charge
- rider count
- fare settlement method
- rate class
- pickup and dropoff locations

The purpose is to determine whether zero-duration records appear
to be:

1. genuine short/instantaneous records,
2. timestamp-related anomalies,
3. cancelled or otherwise unusual trips, or
4. records that contain enough contradictory information to justify
   exclusion.

No records are removed or modified at this stage.

In [53]:
# ============================================================
# 1.7.7.1 Profile Zero-Duration Trips
# ============================================================

zero_duration_trips = taxi_inspection_sample[
    zero_duration
].copy()

print("=" * 70)
print("ZERO-DURATION TRIP PROFILE")
print("=" * 70)

print(f"Zero-duration trips: {len(zero_duration_trips):,}")
print(
    f"Percentage of inspection sample: "
    f"{len(zero_duration_trips) / len(taxi_inspection_sample) * 100:.3f}%"
)

print("\nNumerical characteristics:")

zero_duration_profile = zero_duration_trips[
    [
        "rider_count",
        "distance_miles",
        "base_fare",
        "charge_total",
        "toll_total",
        "driver_tip_payment"
    ]
].describe().T

display(zero_duration_profile)

ZERO-DURATION TRIP PROFILE
Zero-duration trips: 804
Percentage of inspection sample: 0.804%

Numerical characteristics:


,count,mean,std,min,25%,50%,75%,max
rider_count,804.0,1.201493,0.591932,0.0,1.00,1.00,1.00,6.00
distance_miles,804.0,2.347400,3.044738,0.0,0.90,1.43,2.34,30.39
base_fare,804.0,15.444602,12.784833,3.0,8.60,12.10,17.70,147.90
charge_total,804.0,24.865908,16.930834,4.5,16.02,20.35,26.94,153.65
toll_total,804.0,0.321542,1.690890,0.0,0.00,0.00,0.00,16.05
driver_tip_payment,804.0,3.410659,2.991733,0.0,2.00,3.03,4.35,22.90


In [56]:
# ============================================================
# 1.7.7.2 Zero-Duration Trips by Categorical Variables
# ============================================================

categorical_columns = [
    "provider_code",
    "rate_class_id",
    "fare_settlement_method",
    "offline_record_flag"
]

for column in categorical_columns:

    print("\n" + "=" * 70)
    print(f"ZERO-DURATION DISTRIBUTION: {column}")
    print("=" * 70)

    # Distribution among zero-duration trips
    zero_counts = (
        zero_duration_trips[column]
        .value_counts(dropna=False)
        .sort_index()
    )

    # Distribution among all inspection records
    overall_counts = (
        taxi_inspection_sample[column]
        .value_counts(dropna=False)
        .sort_index()
    )

    comparison = pd.DataFrame({
        "zero_duration_count": zero_counts,
        "zero_duration_percentage": (
            zero_counts / len(zero_duration_trips) * 100
        ),
        "overall_count": overall_counts,
        "overall_percentage": (
            overall_counts / len(taxi_inspection_sample) * 100
        )
    }).fillna(0)

    display(comparison)


ZERO-DURATION DISTRIBUTION: provider_code


,zero_duration_count,zero_duration_percentage,overall_count,overall_percentage
provider_code,,,,
1,22,2.736318,22206,22.206
2,2,0.248756,77014,77.014
7,780,97.014925,780,0.780



ZERO-DURATION DISTRIBUTION: rate_class_id


,zero_duration_count,zero_duration_percentage,overall_count,overall_percentage
rate_class_id,,,,
1.0,790.0,98.258706,93534,93.534
2.0,6.0,0.746269,3426,3.426
3.0,3.0,0.373134,289,0.289
4.0,1.0,0.124378,275,0.275
5.0,1.0,0.124378,840,0.840
6.0,0.0,0.000000,1,0.001
99.0,3.0,0.373134,1635,1.635



ZERO-DURATION DISTRIBUTION: fare_settlement_method


,zero_duration_count,zero_duration_percentage,overall_count,overall_percentage
fare_settlement_method,,,,
1,687,85.447761,84240,84.240
2,108,13.432836,12486,12.486
3,8,0.995025,777,0.777
4,1,0.124378,2497,2.497



ZERO-DURATION DISTRIBUTION: offline_record_flag


,zero_duration_count,zero_duration_percentage,overall_count,overall_percentage
offline_record_flag,,,,
N,787,97.885572,99751,99.751
Y,17,2.114428,249,0.249


In [62]:
# ============================================================
# 1.7.7.2a Zero-Duration Trips by Provider
# ============================================================

column = "provider_code"

# Count each provider among zero-duration trips
zero_counts = (
    zero_duration_trips[column]
    .value_counts(dropna=False)
    .sort_index()
)

# Count each provider in the complete inspection sample
overall_counts = (
    taxi_inspection_sample[column]
    .value_counts(dropna=False)
    .sort_index()
)

# Create comparison table
provider_comparison = pd.DataFrame({
    "zero_duration_count": zero_counts,
    "zero_duration_percentage": (
        zero_counts / len(zero_duration_trips) * 100
    ),
    "overall_count": overall_counts,
    "overall_percentage": (
        overall_counts / len(taxi_inspection_sample) * 100
    )
}).fillna(0)

print("=" * 70)
print("ZERO-DURATION TRIPS BY PROVIDER")
print("=" * 70)

display(provider_comparison)

ZERO-DURATION TRIPS BY PROVIDER


,zero_duration_count,zero_duration_percentage,overall_count,overall_percentage
provider_code,,,,
1,22,2.736318,22206,22.206
2,2,0.248756,77014,77.014
7,780,97.014925,780,0.780


### 1.7.7.3 Provider 7 Temporal Pattern Investigation

The previous analysis revealed that all 780 records belonging to
provider_code = 7 in the inspection sample have zero trip duration.

This is a strong indication that the temporal pattern may be
systematic rather than randomly distributed.

We therefore isolate provider 7 and compare its trip characteristics
with providers 1 and 2.

The investigation focuses on:

- trip duration
- distance
- base fare
- charge total
- rider count
- rate class
- fare settlement method
- offline record flag

The objective is to determine whether provider 7 represents a
distinct type of record or whether its timestamps appear anomalous.

No records are removed or modified at this stage.

In [65]:
# ============================================================
# 1.7.7.3 Provider 7 Temporal Pattern Investigation
# ============================================================

# Create a temporary duration column for comparison
provider_analysis = taxi_inspection_sample.copy()

provider_analysis["trip_duration_seconds"] = trip_duration_seconds

# Summarize providers separately
provider_summary = (
    provider_analysis
    .groupby("provider_code")
    .agg(
        trip_count=("provider_code", "size"),
        zero_duration_count=(
            "trip_duration_seconds",
            lambda x: (x == 0).sum()
        ),
        median_duration_seconds=(
            "trip_duration_seconds",
            "median"
        ),
        mean_duration_seconds=(
            "trip_duration_seconds",
            "mean"
        ),
        median_distance_miles=(
            "distance_miles",
            "median"
        ),
        median_base_fare=(
            "base_fare",
            "median"
        ),
        median_charge_total=(
            "charge_total",
            "median"
        ),
        median_rider_count=(
            "rider_count",
            "median"
        )
    )
    .reset_index()
)

# Calculate percentage of each provider's records with zero duration
provider_summary["zero_duration_percentage"] = (
    provider_summary["zero_duration_count"]
    / provider_summary["trip_count"]
    * 100
)

print("=" * 70)
print("PROVIDER-LEVEL TEMPORAL COMPARISON")
print("=" * 70)

display(provider_summary)

PROVIDER-LEVEL TEMPORAL COMPARISON


,provider_code,trip_count,zero_duration_count,median_duration_seconds,mean_duration_seconds,median_distance_miles,median_base_fare,median_charge_total,median_rider_count,zero_duration_percentage
0,1,22206,22,791.0,1083.576916,1.700,12.8,21.060,1.0,0.099072
1,2,77014,2,735.0,982.085906,1.640,12.8,21.450,1.0,0.002597
2,7,780,780,0.0,0.000000,1.455,12.1,20.725,1.0,100.000000


### 1.7.7.4 Provider 7 Record Characteristics

Provider 7 shows a distinctive temporal pattern: every provider 7
record in the inspection sample has zero trip duration.

We therefore examine the categorical characteristics of provider 7
records to determine whether they consistently belong to particular:

- rate classes
- fare settlement methods
- offline record statuses

We also compare their distance and fare characteristics.

This analysis is exploratory. No records are removed or modified.

In [68]:
# ============================================================
# 1.7.7.4 Provider 7 Record Characteristics
# ============================================================

provider_7 = taxi_inspection_sample[
    taxi_inspection_sample["provider_code"] == 7
].copy()

print("=" * 70)
print("PROVIDER 7 RECORD CHARACTERISTICS")
print("=" * 70)

print(f"Provider 7 records: {len(provider_7):,}")

print("\n" + "-" * 70)
print("RATE CLASS DISTRIBUTION")
print("-" * 70)

rate_class_summary = (
    provider_7["rate_class_id"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

rate_class_summary["percentage"] = (
    rate_class_summary["count"]
    / len(provider_7)
    * 100
)

display(rate_class_summary)

print("\n" + "-" * 70)
print("FARE SETTLEMENT METHOD DISTRIBUTION")
print("-" * 70)

settlement_summary = (
    provider_7["fare_settlement_method"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

settlement_summary["percentage"] = (
    settlement_summary["count"]
    / len(provider_7)
    * 100
)

display(settlement_summary)

print("\n" + "-" * 70)
print("OFFLINE RECORD FLAG DISTRIBUTION")
print("-" * 70)

offline_summary = (
    provider_7["offline_record_flag"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

offline_summary["percentage"] = (
    offline_summary["count"]
    / len(provider_7)
    * 100
)

display(offline_summary)

PROVIDER 7 RECORD CHARACTERISTICS
Provider 7 records: 780

----------------------------------------------------------------------
RATE CLASS DISTRIBUTION
----------------------------------------------------------------------


,count,percentage
rate_class_id,,
1.0,770,98.717949
2.0,6,0.769231
3.0,3,0.384615
4.0,1,0.128205



----------------------------------------------------------------------
FARE SETTLEMENT METHOD DISTRIBUTION
----------------------------------------------------------------------


,count,percentage
fare_settlement_method,,
1,682,87.435897
2,89,11.410256
3,8,1.025641
4,1,0.128205



----------------------------------------------------------------------
OFFLINE RECORD FLAG DISTRIBUTION
----------------------------------------------------------------------


,count,percentage
offline_record_flag,,
N,774,99.230769
Y,6,0.769231


In [70]:
# ============================================================
# 1.7.7.5 Inspect Provider 7 Timestamp and Trip Patterns
# ============================================================

comparison_columns = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

# Select provider 7 records
provider_7_examples = (
    taxi_inspection_sample[
        taxi_inspection_sample["provider_code"] == 7
    ][comparison_columns]
    .copy()
)

# Add calculated duration
provider_7_examples["trip_duration_seconds"] = (
    pd.to_datetime(provider_7_examples["dropoff_timestamp"])
    - pd.to_datetime(provider_7_examples["pickup_timestamp"])
).dt.total_seconds()

print("=" * 70)
print("PROVIDER 7 REPRESENTATIVE RECORDS")
print("=" * 70)

display(
    provider_7_examples.head(20)
)

PROVIDER 7 REPRESENTATIVE RECORDS


,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,fare_settlement_method,base_fare,charge_total,trip_duration_seconds
5,7,2025-04-01 00:39:00,2025-04-01 00:39:00,1.0,0.95,1.0,1,5.8,11.55,0.0
46,7,2025-04-01 00:25:00,2025-04-01 00:25:00,1.0,2.04,1.0,2,12.1,17.85,0.0
322,7,2025-04-01 00:04:00,2025-04-01 00:04:00,1.0,0.68,1.0,2,6.5,12.25,0.0
343,7,2025-04-01 00:31:00,2025-04-01 00:31:00,1.0,9.52,1.0,1,40.8,72.29,0.0
566,7,2025-04-01 00:27:00,2025-04-01 00:27:00,1.0,8.79,1.0,1,35.2,45.95,0.0
1149,7,2025-04-01 00:15:00,2025-04-01 00:15:00,1.0,16.95,2.0,1,70.0,100.13,0.0
1150,7,2025-04-01 00:27:00,2025-04-01 00:27:00,1.0,0.66,1.0,1,5.1,13.02,0.0
2102,7,2025-04-01 01:29:00,2025-04-01 01:29:00,1.0,8.63,1.0,1,33.1,50.82,0.0
2481,7,2025-04-01 02:09:00,2025-04-01 02:09:00,1.0,9.02,1.0,1,37.3,68.09,0.0
2871,7,2025-04-01 02:45:00,2025-04-01 02:45:00,1.0,17.91,1.0,1,70.9,90.40,0.0


### 1.7.7.6 Zero Duration vs Distance

The provider-level investigation showed that provider 7 records have
zero duration while retaining nonzero trip distances and fares.

We now quantify how many zero-duration records have:

- zero distance
- positive distance

A trip with zero recorded duration but positive distance is an
important consistency issue because distance indicates that movement
was recorded even though no elapsed travel time was recorded.

This analysis is performed on the inspection sample only.

No records are removed or modified at this stage.

In [73]:
# ============================================================
# 1.7.7.6 Zero Duration vs Distance
# ============================================================

zero_duration_distance_zero = (
    zero_duration_trips["distance_miles"] == 0
)

zero_duration_distance_positive = (
    zero_duration_trips["distance_miles"] > 0
)

zero_distance_count = zero_duration_distance_zero.sum()
positive_distance_count = zero_duration_distance_positive.sum()

distance_consistency_summary = pd.DataFrame({
    "distance_category": [
        "Zero distance",
        "Positive distance"
    ],
    "count": [
        zero_distance_count,
        positive_distance_count
    ]
})

distance_consistency_summary["percentage"] = (
    distance_consistency_summary["count"]
    / len(zero_duration_trips)
    * 100
)

print("=" * 70)
print("ZERO-DURATION TRIPS VS DISTANCE")
print("=" * 70)

display(distance_consistency_summary)

ZERO-DURATION TRIPS VS DISTANCE


,distance_category,count,percentage
0,Zero distance,24,2.985075
1,Positive distance,780,97.014925


## 1.8 Full Dataset Temporal Anomaly Quantification

The inspection sample identified a strong temporal inconsistency
associated with provider_code = 7.

We now evaluate the complete taxi dataset rather than relying on the
100,000-row inspection sample.

Because the taxi dataset contains multiple large monthly CSV files,
the analysis is performed file-by-file. Only the columns required for
the temporal audit are loaded into memory.

For each monthly file, we calculate:

- total records
- negative-duration records
- zero-duration records
- zero-duration records with zero distance
- zero-duration records with positive distance
- provider 7 records
- provider 7 records with zero duration

No records are removed or modified during this audit.

In [76]:
# ============================================================
# 1.8.1 Full Dataset Temporal Anomaly Quantification
# ============================================================

from pathlib import Path
import pandas as pd

# Columns required for this audit only
TEMPORAL_AUDIT_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "distance_miles"
]

monthly_results = []

print("=" * 70)
print("FULL DATASET TEMPORAL ANOMALY AUDIT")
print("=" * 70)

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"\nProcessing: {file_path.name}")

    # Load only the columns required for this audit
    df = pd.read_csv(
        file_path,
        usecols=TEMPORAL_AUDIT_COLUMNS
    )

    # Parse timestamps
    pickup_dt = pd.to_datetime(
        df["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df["dropoff_timestamp"],
        errors="coerce"
    )

    # Calculate duration in seconds
    duration_seconds = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    # Define anomaly conditions
    negative_duration_mask = duration_seconds < 0
    zero_duration_mask = duration_seconds == 0

    zero_duration_zero_distance_mask = (
        zero_duration_mask
        & (df["distance_miles"] == 0)
    )

    zero_duration_positive_distance_mask = (
        zero_duration_mask
        & (df["distance_miles"] > 0)
    )

    provider_7_mask = df["provider_code"] == 7

    provider_7_zero_duration_mask = (
        provider_7_mask
        & zero_duration_mask
    )

    # Store monthly results
    monthly_results.append({
        "file": file_path.name,
        "total_rows": len(df),
        "negative_duration_count": negative_duration_mask.sum(),
        "zero_duration_count": zero_duration_mask.sum(),
        "zero_duration_zero_distance_count":
            zero_duration_zero_distance_mask.sum(),
        "zero_duration_positive_distance_count":
            zero_duration_positive_distance_mask.sum(),
        "provider_7_count": provider_7_mask.sum(),
        "provider_7_zero_duration_count":
            provider_7_zero_duration_mask.sum()
    })

    print(f"Rows processed: {len(df):,}")

# Convert results to DataFrame
temporal_audit_by_file = pd.DataFrame(monthly_results)

print("\n" + "=" * 70)
print("MONTHLY TEMPORAL AUDIT RESULTS")
print("=" * 70)

display(temporal_audit_by_file)

FULL DATASET TEMPORAL ANOMALY AUDIT

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Rows processed: 3,970,553

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Rows processed: 4,591,845

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Rows processed: 4,322,960

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Rows processed: 3,898,963

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Rows processed: 3,574,091

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Rows processed: 4,251,015

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Rows processed: 4,428,699

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Rows processed: 4,181,444

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Rows processed: 4,305,006

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Rows processed: 3,724,889

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Rows processed: 3,399,866

Processing: Urban_Flow_Analy

,file,total_rows,negative_duration_count,zero_duration_count,zero_duration_zero_distance_count,zero_duration_positive_distance_count,provider_7_count,provider_7_zero_duration_count
0,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,3970553,163,34606,836,33770,33844,33844
1,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,4591845,101,64171,1522,62649,63261,63261
2,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,4322960,231,68352,2014,66338,67573,67573
3,Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv,3898963,1,56063,1509,54554,55438,55438
4,Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,3574091,2,47831,1665,46166,47276,47276
5,Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv,4251015,0,57179,1203,55976,56583,56583
6,Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,4428699,2,67966,1333,66633,67360,67360
7,Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,4181444,1435,60685,1282,59403,60043,60043
8,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,4305006,2,58019,2055,55964,57416,57416
9,Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv,3724889,1,45069,1164,43905,44705,44705


### 1.8.2 Overall Temporal Anomaly Rates

The monthly audit results are aggregated across the complete taxi dataset.

For each anomaly type, we calculate:

- total number of records
- anomaly count
- percentage of the complete dataset affected

These percentages will support the final data-quality treatment decisions.

In [79]:
# ============================================================
# 1.8.2 Overall Temporal Anomaly Rates
# ============================================================

# Aggregate the complete dataset
total_rows = temporal_audit_by_file["total_rows"].sum()

total_negative_duration = (
    temporal_audit_by_file["negative_duration_count"].sum()
)

total_zero_duration = (
    temporal_audit_by_file["zero_duration_count"].sum()
)

total_zero_duration_zero_distance = (
    temporal_audit_by_file[
        "zero_duration_zero_distance_count"
    ].sum()
)

total_zero_duration_positive_distance = (
    temporal_audit_by_file[
        "zero_duration_positive_distance_count"
    ].sum()
)

total_provider_7 = (
    temporal_audit_by_file["provider_7_count"].sum()
)

total_provider_7_zero_duration = (
    temporal_audit_by_file[
        "provider_7_zero_duration_count"
    ].sum()
)


# ------------------------------------------------------------
# Calculate percentages
# ------------------------------------------------------------

negative_duration_pct = (
    total_negative_duration / total_rows * 100
)

zero_duration_pct = (
    total_zero_duration / total_rows * 100
)

zero_duration_zero_distance_pct = (
    total_zero_duration_zero_distance / total_rows * 100
)

zero_duration_positive_distance_pct = (
    total_zero_duration_positive_distance / total_rows * 100
)

provider_7_pct = (
    total_provider_7 / total_rows * 100
)

provider_7_zero_duration_pct = (
    total_provider_7_zero_duration / total_rows * 100
)


# ------------------------------------------------------------
# Create summary table
# ------------------------------------------------------------

temporal_summary = pd.DataFrame({
    "anomaly_type": [
        "Negative duration",
        "Zero duration",
        "Zero duration + zero distance",
        "Zero duration + positive distance",
        "Provider 7",
        "Provider 7 + zero duration"
    ],
    "count": [
        total_negative_duration,
        total_zero_duration,
        total_zero_duration_zero_distance,
        total_zero_duration_positive_distance,
        total_provider_7,
        total_provider_7_zero_duration
    ],
    "percentage_of_total": [
        negative_duration_pct,
        zero_duration_pct,
        zero_duration_zero_distance_pct,
        zero_duration_positive_distance_pct,
        provider_7_pct,
        provider_7_zero_duration_pct
    ]
})


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("FULL DATASET TEMPORAL ANOMALY SUMMARY")
print("=" * 70)

print(f"\nTotal taxi records: {total_rows:,}")

display(
    temporal_summary.style.format({
        "percentage_of_total": "{:.4f}%"
    })
)

FULL DATASET TEMPORAL ANOMALY SUMMARY

Total taxi records: 48,601,782


,anomaly_type,count,percentage_of_total
0,Negative duration,1942,0.0040%
1,Zero duration,649668,1.3367%
2,Zero duration + zero distance,17043,0.0351%
3,Zero duration + positive distance,632625,1.3016%
4,Provider 7,642536,1.3220%
5,Provider 7 + zero duration,642536,1.3220%


### 1.8.3 Investigation of Negative-Duration Records

Negative trip duration is explicitly identified as a data-quality anomaly.

Before deciding whether these records should be removed, we examine their
characteristics across:

- month
- provider
- magnitude of the negative duration
- distance
- rider count
- base fare
- charge total
- settlement method
- rate class
- offline record status

The purpose is to determine whether the anomaly is random or associated
with a systematic recording pattern.

In [84]:
# ============================================================
# 1.8.3 Investigate Negative-Duration Records
# ============================================================

NEGATIVE_AUDIT_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

negative_results = []

print("=" * 70)
print("NEGATIVE-DURATION INVESTIGATION")
print("=" * 70)

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"\nProcessing: {file_path.name}")

    df_neg = pd.read_csv(
        file_path,
        usecols=NEGATIVE_AUDIT_COLUMNS
    )

    # Parse timestamps
    pickup_dt = pd.to_datetime(
        df_neg["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df_neg["dropoff_timestamp"],
        errors="coerce"
    )

    # Calculate trip duration in seconds
    duration_seconds = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    # Identify negative-duration records
    negative_mask = duration_seconds < 0

    if negative_mask.sum() == 0:
        continue

    # Keep only negative-duration records
    neg = df_neg.loc[negative_mask].copy()

    # Add duration information
    neg["duration_seconds"] = duration_seconds.loc[negative_mask]

    # Convert negative duration into a positive magnitude
    neg["negative_duration_seconds"] = (
        -neg["duration_seconds"]
    )

    # Keep source file for later analysis
    neg["source_file"] = file_path.name

    negative_results.append(neg)


# ------------------------------------------------------------
# Combine all negative-duration records
# ------------------------------------------------------------

negative_duration_df = pd.concat(
    negative_results,
    ignore_index=True
)


# ------------------------------------------------------------
# Display basic results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NEGATIVE-DURATION RECORDS FOUND")
print("=" * 70)

print(
    f"Total negative-duration records: "
    f"{len(negative_duration_df):,}"
)

print("\nNegative duration statistics:")

display(
    negative_duration_df[
        [
            "duration_seconds",
            "negative_duration_seconds"
        ]
    ].describe()
)

NEGATIVE-DURATION INVESTIGATION

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_13152\391413114.py:28: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_neg = pd.read_csv(



Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_13152\391413114.py:28: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_neg = pd.read_csv(



Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv


...

C:\Users\arudk\AppData\Local\Temp\ipykernel_13152\391413114.py:28: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_neg = pd.read_csv(



Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_13152\391413114.py:28: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_neg = pd.read_csv(



NEGATIVE-DURATION RECORDS FOUND
Total negative-duration records: 1,942

Negative duration statistics:


,duration_seconds,negative_duration_seconds
count,1942.000000,1942.000000
mean,-1857.239959,1857.239959
std,1861.474929,1861.474929
min,-42439.000000,1.000000
25%,-2699.750000,151.500000
50%,-2210.000000,2210.000000
75%,-151.500000,2699.750000
max,-1.000000,42439.000000


### 1.8.4 Distribution of Negative-Duration Records

The negative-duration records are further examined by:

- source month
- provider
- offline record status
- settlement method
- rate class

This helps determine whether negative durations represent isolated
timestamp errors or a systematic data-recording pattern.

No records are removed or modified at this stage.

In [87]:
# ============================================================
# 1.8.4 Negative-Duration Distribution
# ============================================================

print("=" * 70)
print("NEGATIVE-DURATION DISTRIBUTION")
print("=" * 70)


# ------------------------------------------------------------
# 1. By source month
# ------------------------------------------------------------

print("\n1. NEGATIVE DURATIONS BY MONTH")
print("-" * 70)

negative_by_month = (
    negative_duration_df
    .groupby("source_file")
    .size()
    .reset_index(name="negative_duration_count")
)

negative_by_month["percentage_of_negative_records"] = (
    negative_by_month["negative_duration_count"]
    / len(negative_duration_df)
    * 100
)

display(
    negative_by_month.style.format({
        "percentage_of_negative_records": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 2. By provider
# ------------------------------------------------------------

print("\n2. NEGATIVE DURATIONS BY PROVIDER")
print("-" * 70)

negative_by_provider = (
    negative_duration_df
    .groupby("provider_code")
    .size()
    .reset_index(name="negative_duration_count")
)

negative_by_provider["percentage_of_negative_records"] = (
    negative_by_provider["negative_duration_count"]
    / len(negative_duration_df)
    * 100
)

display(
    negative_by_provider.style.format({
        "percentage_of_negative_records": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 3. By offline record flag
# ------------------------------------------------------------

print("\n3. NEGATIVE DURATIONS BY OFFLINE RECORD STATUS")
print("-" * 70)

negative_by_offline = (
    negative_duration_df
    .groupby("offline_record_flag")
    .size()
    .reset_index(name="negative_duration_count")
)

negative_by_offline["percentage_of_negative_records"] = (
    negative_by_offline["negative_duration_count"]
    / len(negative_duration_df)
    * 100
)

display(
    negative_by_offline.style.format({
        "percentage_of_negative_records": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 4. By settlement method
# ------------------------------------------------------------

print("\n4. NEGATIVE DURATIONS BY SETTLEMENT METHOD")
print("-" * 70)

negative_by_settlement = (
    negative_duration_df
    .groupby("fare_settlement_method")
    .size()
    .reset_index(name="negative_duration_count")
)

negative_by_settlement["percentage_of_negative_records"] = (
    negative_by_settlement["negative_duration_count"]
    / len(negative_duration_df)
    * 100
)

display(
    negative_by_settlement.style.format({
        "percentage_of_negative_records": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 5. By rate class
# ------------------------------------------------------------

print("\n5. NEGATIVE DURATIONS BY RATE CLASS")
print("-" * 70)

negative_by_rate_class = (
    negative_duration_df
    .groupby("rate_class_id")
    .size()
    .reset_index(name="negative_duration_count")
)

negative_by_rate_class["percentage_of_negative_records"] = (
    negative_by_rate_class["negative_duration_count"]
    / len(negative_duration_df)
    * 100
)

display(
    negative_by_rate_class.style.format({
        "percentage_of_negative_records": "{:.2f}%"
    })
)

NEGATIVE-DURATION DISTRIBUTION

1. NEGATIVE DURATIONS BY MONTH
----------------------------------------------------------------------


,source_file,negative_duration_count,percentage_of_negative_records
0,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,163,8.39%
1,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,101,5.20%
2,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,231,11.89%
3,Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv,1,0.05%
4,Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,2,0.10%
5,Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,2,0.10%
6,Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,1435,73.89%
7,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,2,0.10%
8,Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv,1,0.05%
9,Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv,2,0.10%



2. NEGATIVE DURATIONS BY PROVIDER
----------------------------------------------------------------------


,provider_code,negative_duration_count,percentage_of_negative_records
0,1,349,17.97%
1,2,1113,57.31%
2,6,480,24.72%



3. NEGATIVE DURATIONS BY OFFLINE RECORD STATUS
----------------------------------------------------------------------


,offline_record_flag,negative_duration_count,percentage_of_negative_records
0,N,1031,53.09%
1,Y,8,0.41%



4. NEGATIVE DURATIONS BY SETTLEMENT METHOD
----------------------------------------------------------------------


,fare_settlement_method,negative_duration_count,percentage_of_negative_records
0,0,903,46.50%
1,1,943,48.56%
2,2,74,3.81%
3,3,6,0.31%
4,4,16,0.82%



5. NEGATIVE DURATIONS BY RATE CLASS
----------------------------------------------------------------------


,rate_class_id,negative_duration_count,percentage_of_negative_records
0,1.000000,988,50.88%
1,2.000000,7,0.36%
2,4.000000,2,0.10%
3,5.000000,14,0.72%
4,99.000000,28,1.44%


### 1.8.5 Missing-Value Investigation Within Negative Durations

The previous distribution analysis showed that the offline-record and
rate-class groups did not account for all 1,942 negative-duration
records.

We therefore explicitly investigate missing values within the
negative-duration subset.

This prevents missing categories from being silently excluded from
the anomaly analysis.

In [90]:
# ============================================================
# 1.8.5 Missing-Value Investigation Within Negative Durations
# ============================================================

print("=" * 70)
print("MISSING VALUES WITHIN NEGATIVE-DURATION RECORDS")
print("=" * 70)

total_negative = len(negative_duration_df)

# ------------------------------------------------------------
# 1. Missing-value summary
# ------------------------------------------------------------

missing_negative = (
    negative_duration_df.isna()
    .sum()
    .reset_index()
)

missing_negative.columns = [
    "column",
    "missing_count"
]

missing_negative["missing_percentage"] = (
    missing_negative["missing_count"]
    / total_negative
    * 100
)

missing_negative = missing_negative[
    missing_negative["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)

print("\n1. Missing values among negative-duration records")
print("-" * 70)

display(
    missing_negative.style.format({
        "missing_percentage": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 2. Explicitly include missing categories
# ------------------------------------------------------------

print("\n2. Offline-record status including missing values")
print("-" * 70)

offline_distribution = (
    negative_duration_df["offline_record_flag"]
    .fillna("<MISSING>")
    .value_counts(dropna=False)
    .rename_axis("offline_record_flag")
    .reset_index(name="count")
)

offline_distribution["percentage"] = (
    offline_distribution["count"]
    / total_negative
    * 100
)

display(
    offline_distribution.style.format({
        "percentage": "{:.2f}%"
    })
)


print("\n3. Rate class including missing values")
print("-" * 70)

rate_distribution = (
    negative_duration_df["rate_class_id"]
    .fillna(-1)
    .value_counts(dropna=False)
    .rename_axis("rate_class_id")
    .reset_index(name="count")
)

rate_distribution["rate_class_label"] = (
    rate_distribution["rate_class_id"]
    .apply(
        lambda x: "<MISSING>"
        if x == -1
        else str(x)
    )
)

rate_distribution["percentage"] = (
    rate_distribution["count"]
    / total_negative
    * 100
)

display(
    rate_distribution[
        [
            "rate_class_label",
            "count",
            "percentage"
        ]
    ].style.format({
        "percentage": "{:.2f}%"
    })
)

MISSING VALUES WITHIN NEGATIVE-DURATION RECORDS

1. Missing values among negative-duration records
----------------------------------------------------------------------


,column,missing_count,missing_percentage
3,rider_count,903,46.50%
5,rate_class_id,903,46.50%
6,offline_record_flag,903,46.50%



2. Offline-record status including missing values
----------------------------------------------------------------------


,offline_record_flag,count,percentage
0,N,1031,53.09%
1,,903,46.50%
2,Y,8,0.41%



3. Rate class including missing values
----------------------------------------------------------------------


,rate_class_label,count,percentage
0,1.0,988,50.88%
1,,903,46.50%
2,99.0,28,1.44%
3,5.0,14,0.72%
4,2.0,7,0.36%
5,4.0,2,0.10%


### 1.8.6 Profile of Negative-Duration Records with Missing Fields

A subset of negative-duration records contains missing values in
rider_count, rate_class_id, and offline_record_flag.

We investigate whether these missing values occur together and whether
the affected records are concentrated by:

- month
- provider
- fare settlement method
- duration magnitude
- distance
- fare

This helps identify whether the records represent a systematic
recording pattern.

In [93]:
# ============================================================
# 1.8.6 Profile Negative-Duration Records with Missing Fields
# ============================================================

print("=" * 70)
print("NEGATIVE-DURATION RECORDS WITH MISSING FIELDS")
print("=" * 70)


# ------------------------------------------------------------
# Define records where all three fields are missing
# ------------------------------------------------------------

missing_fields_mask = (
    negative_duration_df["rider_count"].isna()
    & negative_duration_df["rate_class_id"].isna()
    & negative_duration_df["offline_record_flag"].isna()
)

negative_missing_df = negative_duration_df.loc[
    missing_fields_mask
].copy()


# ------------------------------------------------------------
# Check whether all missing values occur together
# ------------------------------------------------------------

print("\n1. Missing-field overlap")
print("-" * 70)

print(
    f"Total negative-duration records: "
    f"{len(negative_duration_df):,}"
)

print(
    f"Records missing all three fields: "
    f"{len(negative_missing_df):,}"
)

print(
    f"Percentage of negative-duration records: "
    f"{len(negative_missing_df) / len(negative_duration_df) * 100:.2f}%"
)


# ------------------------------------------------------------
# 2. Distribution by month
# ------------------------------------------------------------

print("\n2. Missing-field records by month")
print("-" * 70)

missing_by_month = (
    negative_missing_df
    .groupby("source_file")
    .size()
    .reset_index(name="count")
)

missing_by_month["percentage"] = (
    missing_by_month["count"]
    / len(negative_missing_df)
    * 100
)

display(
    missing_by_month.style.format({
        "percentage": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 3. Distribution by provider
# ------------------------------------------------------------

print("\n3. Missing-field records by provider")
print("-" * 70)

missing_by_provider = (
    negative_missing_df
    .groupby("provider_code")
    .size()
    .reset_index(name="count")
)

missing_by_provider["percentage"] = (
    missing_by_provider["count"]
    / len(negative_missing_df)
    * 100
)

display(
    missing_by_provider.style.format({
        "percentage": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 4. Settlement method
# ------------------------------------------------------------

print("\n4. Missing-field records by settlement method")
print("-" * 70)

missing_by_settlement = (
    negative_missing_df
    .groupby("fare_settlement_method")
    .size()
    .reset_index(name="count")
)

missing_by_settlement["percentage"] = (
    missing_by_settlement["count"]
    / len(negative_missing_df)
    * 100
)

display(
    missing_by_settlement.style.format({
        "percentage": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# 5. Numerical characteristics
# ------------------------------------------------------------

print("\n5. Numerical characteristics")
print("-" * 70)

display(
    negative_missing_df[
        [
            "negative_duration_seconds",
            "distance_miles",
            "base_fare",
            "charge_total"
        ]
    ].describe()
)

NEGATIVE-DURATION RECORDS WITH MISSING FIELDS

1. Missing-field overlap
----------------------------------------------------------------------
Total negative-duration records: 1,942
Records missing all three fields: 903
Percentage of negative-duration records: 46.50%

2. Missing-field records by month
----------------------------------------------------------------------


,source_file,count,percentage
0,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,153,16.94%
1,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,94,10.41%
2,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,230,25.47%
3,Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,421,46.62%
4,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,2,0.22%
5,Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv,1,0.11%
6,Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv,2,0.22%



3. Missing-field records by provider
----------------------------------------------------------------------


,provider_code,count,percentage
0,1,183,20.27%
1,2,240,26.58%
2,6,480,53.16%



4. Missing-field records by settlement method
----------------------------------------------------------------------


,fare_settlement_method,count,percentage
0,0,903,100.00%



5. Numerical characteristics
----------------------------------------------------------------------


,negative_duration_seconds,distance_miles,base_fare,charge_total
count,903.000000,903.000000,903.000000,903.000000
mean,989.526024,6.443134,10.766932,24.738328
std,1134.969391,4.470728,14.991626,15.278901
min,1.000000,0.000000,-39.100000,-27.410000
25%,17.500000,2.990000,2.900000,16.000000
50%,45.000000,5.410000,2.900000,24.130000
75%,2071.000000,8.915000,21.700000,34.455000
max,11200.000000,26.060000,71.080000,89.130000


### 1.8.7 Inspect Representative Negative-Duration Records

A representative sample of negative-duration records is inspected
directly to understand the nature of the timestamp inconsistency.

The sample includes both:

- records with very small negative durations
- records with larger negative durations

The original pickup and dropoff timestamps are retained so that the
temporal relationship can be examined directly.

No records are modified or removed.

In [96]:
# ============================================================
# 1.8.7 Inspect Representative Negative-Duration Records
# ============================================================

print("=" * 70)
print("REPRESENTATIVE NEGATIVE-DURATION RECORDS")
print("=" * 70)


# ------------------------------------------------------------
# Create a readable copy
# ------------------------------------------------------------

inspection_df = negative_duration_df.copy()

inspection_df["pickup_dt"] = pd.to_datetime(
    inspection_df["pickup_timestamp"],
    errors="coerce"
)

inspection_df["dropoff_dt"] = pd.to_datetime(
    inspection_df["dropoff_timestamp"],
    errors="coerce"
)


# ------------------------------------------------------------
# Select records from different severity levels
# ------------------------------------------------------------

small_negative = (
    inspection_df[
        inspection_df["negative_duration_seconds"] <= 60
    ]
    .sort_values("negative_duration_seconds")
    .head(10)
)

medium_negative = (
    inspection_df[
        (inspection_df["negative_duration_seconds"] > 60)
        & (inspection_df["negative_duration_seconds"] <= 3600)
    ]
    .sort_values("negative_duration_seconds")
    .iloc[::max(
        1,
        len(
            inspection_df[
                (inspection_df["negative_duration_seconds"] > 60)
                & (inspection_df["negative_duration_seconds"] <= 3600)
            ]
        ) // 10
    )]
    .head(10)
)

large_negative = (
    inspection_df[
        inspection_df["negative_duration_seconds"] > 3600
    ]
    .sort_values("negative_duration_seconds")
    .head(10)
)


# ------------------------------------------------------------
# Combine samples
# ------------------------------------------------------------

representative_negative = pd.concat(
    [
        small_negative,
        medium_negative,
        large_negative
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Display important fields
# ------------------------------------------------------------

DISPLAY_COLUMNS = [
    "source_file",
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "duration_seconds",
    "negative_duration_seconds",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

display(
    representative_negative[DISPLAY_COLUMNS]
)

REPRESENTATIVE NEGATIVE-DURATION RECORDS


,source_file,provider_code,pickup_timestamp,dropoff_timestamp,duration_seconds,negative_duration_seconds,rider_count,distance_miles,rate_class_id,offline_record_flag,fare_settlement_method,base_fare,charge_total
0,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,6,2025-04-17 10:04:57,2025-04-17 10:04:56,-1.0,1.0,NaN,0.73,NaN,NaN,0,2.90,16.00
1,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,6,2025-05-31 13:05:01,2025-05-31 13:05:00,-1.0,1.0,NaN,1.59,NaN,NaN,0,2.90,16.00
2,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,1,2025-04-23 19:58:13,2025-04-23 19:58:12,-1.0,1.0,0.0,0.60,1.0,N,2,4.40,26.53
3,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,6,2025-06-04 15:06:20,2025-06-04 15:06:19,-1.0,1.0,NaN,1.69,NaN,NaN,0,2.90,16.00
4,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,6,2025-04-16 14:04:26,2025-04-16 14:04:25,-1.0,1.0,NaN,2.00,NaN,NaN,0,2.90,16.00
5,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,6,2025-05-29 17:05:54,2025-05-29 17:05:53,-1.0,1.0,NaN,4.31,NaN,NaN,0,2.90,19.38
6,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,1,2025-12-03 09:01:43,2025-12-03 09:01:42,-1.0,1.0,NaN,0.00,NaN,NaN,0,24.86,28.98
7,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,6,2025-06-16 08:06:19,2025-06-16 08:06:18,-1.0,1.0,NaN,1.72,NaN,NaN,0,2.90,16.00
8,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,6,2025-06-20 10:06:19,2025-06-20 10:06:18,-1.0,1.0,NaN,4.46,NaN,NaN,0,2.90,26.20
9,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,6,2025-04-10 00:04:43,2025-04-10 00:04:42,-1.0,1.0,NaN,13.67,NaN,NaN,0,1.45,38.58


### 1.8.8 Zero-Duration Records by Provider

Zero-duration trips represent 1.3367% of the complete taxi dataset.

Because most zero-duration records have positive distance, we investigate
their distribution by provider.

This is particularly important because the inspection sample showed that
provider_code = 7 may have a systematic zero-duration recording pattern.

No records are removed or modified during this analysis.

In [99]:
# ============================================================
# 1.8.8 Zero-Duration Records by Provider
# ============================================================

ZERO_DURATION_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "distance_miles"
]

zero_duration_results = []

print("=" * 70)
print("ZERO-DURATION RECORDS BY PROVIDER")
print("=" * 70)

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"\nProcessing: {file_path.name}")

    df_zero = pd.read_csv(
        file_path,
        usecols=ZERO_DURATION_COLUMNS,
        low_memory=False
    )

    pickup_dt = pd.to_datetime(
        df_zero["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df_zero["dropoff_timestamp"],
        errors="coerce"
    )

    duration_seconds = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    zero_mask = duration_seconds == 0

    zero = df_zero.loc[zero_mask].copy()

    if len(zero) == 0:
        continue

    # Classify zero-duration trips by distance
    zero["distance_category"] = "Zero distance"

    zero.loc[
        zero["distance_miles"] > 0,
        "distance_category"
    ] = "Positive distance"

    # Provider-level aggregation
    provider_summary = (
        zero
        .groupby(
            ["provider_code", "distance_category"]
        )
        .size()
        .reset_index(name="zero_duration_count")
    )

    provider_summary["source_file"] = file_path.name

    zero_duration_results.append(provider_summary)


# ------------------------------------------------------------
# Combine monthly results
# ------------------------------------------------------------

zero_duration_by_provider = pd.concat(
    zero_duration_results,
    ignore_index=True
)


# ------------------------------------------------------------
# Aggregate across all months
# ------------------------------------------------------------

zero_duration_provider_total = (
    zero_duration_by_provider
    .groupby(
        ["provider_code", "distance_category"]
    )["zero_duration_count"]
    .sum()
    .reset_index()
)

zero_duration_provider_total[
    "percentage_of_all_zero_duration"
] = (
    zero_duration_provider_total[
        "zero_duration_count"
    ]
    / total_zero_duration
    * 100
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL ZERO-DURATION DISTRIBUTION BY PROVIDER")
print("=" * 70)

display(
    zero_duration_provider_total.style.format({
        "percentage_of_all_zero_duration": "{:.2f}%"
    })
)

ZERO-DURATION RECORDS BY PROVIDER

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

OVERALL ZERO-DURATION DISTRIBUTION BY PROVIDER


,provider_code,distance_category,zero_duration_count,percentage_of_all_zero_duration
0,1,Positive distance,180,0.03%
1,1,Zero distance,6112,0.94%
2,2,Positive distance,319,0.05%
3,2,Zero distance,503,0.08%
4,6,Positive distance,18,0.00%
5,7,Positive distance,632108,97.30%
6,7,Zero distance,10428,1.61%


### 1.8.9 Provider 7 Temporal Profile

Provider 7 accounts for the overwhelming majority of zero-duration
records.

We therefore examine all Provider 7 trips and compare their duration
distribution with other providers.

The analysis evaluates:

- number of Provider 7 records
- zero-duration records
- positive-duration records
- negative-duration records
- distance distribution
- fare distribution

This determines whether Provider 7 represents a systematic temporal
recording pattern and whether its records can be used for duration-based
modeling.

In [102]:
# ============================================================
# 1.8.9 Provider 7 Temporal Profile
# ============================================================

PROVIDER_PROFILE_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "distance_miles",
    "base_fare",
    "charge_total"
]

provider_profile_results = []

print("=" * 70)
print("PROVIDER TEMPORAL PROFILE")
print("=" * 70)

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"\nProcessing: {file_path.name}")

    df_provider = pd.read_csv(
        file_path,
        usecols=PROVIDER_PROFILE_COLUMNS,
        low_memory=False
    )

    # Parse timestamps
    pickup_dt = pd.to_datetime(
        df_provider["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df_provider["dropoff_timestamp"],
        errors="coerce"
    )

    # Calculate duration
    duration_seconds = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    # Add duration to dataframe
    df_provider["duration_seconds"] = duration_seconds

    # Classify duration
    df_provider["duration_category"] = "Positive"

    df_provider.loc[
        df_provider["duration_seconds"] == 0,
        "duration_category"
    ] = "Zero"

    df_provider.loc[
        df_provider["duration_seconds"] < 0,
        "duration_category"
    ] = "Negative"

    # Keep only Provider 7
    provider_7 = df_provider[
        df_provider["provider_code"] == 7
    ].copy()

    if len(provider_7) == 0:
        continue

    # Aggregate by duration category
    summary = (
        provider_7
        .groupby("duration_category")
        .agg(
            record_count=("duration_seconds", "size"),
            median_duration_seconds=("duration_seconds", "median"),
            mean_distance_miles=("distance_miles", "mean"),
            median_distance_miles=("distance_miles", "median"),
            mean_base_fare=("base_fare", "mean"),
            median_base_fare=("base_fare", "median"),
            mean_charge_total=("charge_total", "mean"),
            median_charge_total=("charge_total", "median")
        )
        .reset_index()
    )

    summary["source_file"] = file_path.name

    provider_profile_results.append(summary)


# ------------------------------------------------------------
# Combine monthly results
# ------------------------------------------------------------

provider_7_profile = pd.concat(
    provider_profile_results,
    ignore_index=True
)


# ------------------------------------------------------------
# Aggregate across all months
# ------------------------------------------------------------

provider_7_overall = (
    provider_7_profile
    .groupby("duration_category")
    .agg(
        record_count=("record_count", "sum"),
        median_duration_seconds=("median_duration_seconds", "median"),
        mean_distance_miles=("mean_distance_miles", "mean"),
        median_distance_miles=("median_distance_miles", "median"),
        mean_base_fare=("mean_base_fare", "mean"),
        median_base_fare=("median_base_fare", "median"),
        mean_charge_total=("mean_charge_total", "mean"),
        median_charge_total=("median_charge_total", "median")
    )
    .reset_index()
)

provider_7_total = provider_7_overall["record_count"].sum()

provider_7_overall["percentage_of_provider_7"] = (
    provider_7_overall["record_count"]
    / provider_7_total
    * 100
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL PROVIDER 7 DURATION PROFILE")
print("=" * 70)

print(
    f"\nTotal Provider 7 records: "
    f"{provider_7_total:,}"
)

display(
    provider_7_overall.style.format({
        "percentage_of_provider_7": "{:.2f}%",
        "median_duration_seconds": "{:.2f}",
        "mean_distance_miles": "{:.2f}",
        "median_distance_miles": "{:.2f}",
        "mean_base_fare": "{:.2f}",
        "median_base_fare": "{:.2f}",
        "mean_charge_total": "{:.2f}",
        "median_charge_total": "{:.2f}"
    })
)

PROVIDER TEMPORAL PROFILE

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

OVERALL PROVIDER 7 DURATION PROFILE

Total Provider 7 records: 642,536


,duration_category,record_count,median_duration_seconds,mean_distance_miles,median_distance_miles,mean_base_fare,median_base_fare,mean_charge_total,median_charge_total,percentage_of_provider_7
0,Zero,642536,0.00,2.73,1.56,16.76,12.10,26.25,20.58,100.00%


### 1.8.10 Non-Provider-7 Zero-Duration Records

Provider 7 accounts for 98.90% of all zero-duration records.

The remaining zero-duration records are investigated separately to
determine whether they represent valid records, timestamp errors, or
other data-quality issues.

The analysis compares:

- provider
- zero vs positive distance
- distance
- fare
- charge total

No records are removed or modified during this analysis.

In [105]:
# ============================================================
# 1.8.10 Non-Provider-7 Zero-Duration Records
# ============================================================

NON_PROVIDER7_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "distance_miles",
    "base_fare",
    "charge_total"
]

non_provider7_zero_results = []

print("=" * 70)
print("NON-PROVIDER-7 ZERO-DURATION RECORDS")
print("=" * 70)

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"\nProcessing: {file_path.name}")

    df_z = pd.read_csv(
        file_path,
        usecols=NON_PROVIDER7_COLUMNS,
        low_memory=False
    )

    pickup_dt = pd.to_datetime(
        df_z["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df_z["dropoff_timestamp"],
        errors="coerce"
    )

    duration_seconds = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    # Zero duration AND not Provider 7
    mask = (
        (duration_seconds == 0)
        & (df_z["provider_code"] != 7)
    )

    zero_non7 = df_z.loc[mask].copy()

    if len(zero_non7) == 0:
        continue

    # Classify distance
    zero_non7["distance_category"] = "Zero distance"

    zero_non7.loc[
        zero_non7["distance_miles"] > 0,
        "distance_category"
    ] = "Positive distance"

    zero_non7["source_file"] = file_path.name

    non_provider7_zero_results.append(
        zero_non7
    )


# ------------------------------------------------------------
# Combine all months
# ------------------------------------------------------------

non_provider7_zero_df = pd.concat(
    non_provider7_zero_results,
    ignore_index=True
)

print("\n" + "=" * 70)
print("TOTAL NON-PROVIDER-7 ZERO-DURATION RECORDS")
print("=" * 70)

print(
    f"Total records: "
    f"{len(non_provider7_zero_df):,}"
)


# ------------------------------------------------------------
# Distribution by provider and distance
# ------------------------------------------------------------

print("\n1. Distribution by provider and distance")
print("-" * 70)

non_provider7_distribution = (
    non_provider7_zero_df
    .groupby(
        ["provider_code", "distance_category"]
    )
    .size()
    .reset_index(name="count")
)

non_provider7_distribution["percentage"] = (
    non_provider7_distribution["count"]
    / len(non_provider7_zero_df)
    * 100
)

display(
    non_provider7_distribution.style.format({
        "percentage": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# Numerical profile
# ------------------------------------------------------------

print("\n2. Numerical profile")
print("-" * 70)

display(
    non_provider7_zero_df[
        [
            "distance_miles",
            "base_fare",
            "charge_total"
        ]
    ].describe()
)

NON-PROVIDER-7 ZERO-DURATION RECORDS

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

TOTAL NON-PROVIDER-7 ZERO-DURATION RECORDS
Total records: 7,132

1. Distribution by provider and distance
----------------------------------------------------------------------


,provider_code,distance_category,count,percentage
0,1,Positive distance,180,2.52%
1,1,Zero distance,6112,85.70%
2,2,Positive distance,319,4.47%
3,2,Zero distance,503,7.05%
4,6,Positive distance,18,0.25%



2. Numerical profile
----------------------------------------------------------------------


,distance_miles,base_fare,charge_total
count,7132.000000,7132.000000,7132.000000
mean,3.278741,24.558252,28.657583
std,261.056481,51.720452,52.268689
min,0.000000,-13.560000,-8.560000
25%,0.000000,3.700000,9.500000
50%,0.000000,14.900000,18.200000
75%,0.000000,27.605000,30.000000
max,22046.450000,2500.000000,2500.000000


### 1.8.11 Zero-Distance Trips with Nonzero Fare

The challenge explicitly requires identification and treatment of trips
where distance_miles = 0 while a nonzero fare is recorded.

This audit examines the complete taxi dataset and quantifies:

- zero-distance records
- zero-distance records with nonzero base fare
- zero-distance records with nonzero charge total
- provider distribution
- duration distribution

No records are removed or modified during this audit.

In [108]:
# ============================================================
# 1.8.11 Zero-Distance + Nonzero-Fare Audit
# ============================================================

ZERO_DISTANCE_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "distance_miles",
    "base_fare",
    "charge_total"
]

zero_distance_results = []

print("=" * 70)
print("ZERO-DISTANCE + NONZERO-FARE AUDIT")
print("=" * 70)

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"\nProcessing: {file_path.name}")

    df_zd = pd.read_csv(
        file_path,
        usecols=ZERO_DISTANCE_COLUMNS,
        low_memory=False
    )

    # Parse timestamps
    pickup_dt = pd.to_datetime(
        df_zd["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df_zd["dropoff_timestamp"],
        errors="coerce"
    )

    # Calculate duration
    duration_seconds = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    df_zd["duration_seconds"] = duration_seconds

    # Zero-distance records
    zero_distance_mask = (
        df_zd["distance_miles"] == 0
    )

    zero_distance = df_zd.loc[
        zero_distance_mask
    ].copy()

    if len(zero_distance) == 0:
        continue

    # Classify fare status
    zero_distance["fare_category"] = "Zero base fare"

    zero_distance.loc[
        zero_distance["base_fare"] != 0,
        "fare_category"
    ] = "Nonzero base fare"

    # Classify charge status
    zero_distance["charge_category"] = "Zero charge total"

    zero_distance.loc[
        zero_distance["charge_total"] != 0,
        "charge_category"
    ] = "Nonzero charge total"

    # Source file
    zero_distance["source_file"] = file_path.name

    zero_distance_results.append(
        zero_distance
    )


# ------------------------------------------------------------
# Combine all months
# ------------------------------------------------------------

zero_distance_df = pd.concat(
    zero_distance_results,
    ignore_index=True
)

print("\n" + "=" * 70)
print("OVERALL ZERO-DISTANCE RESULTS")
print("=" * 70)

print(
    f"Total zero-distance records: "
    f"{len(zero_distance_df):,}"
)


# ------------------------------------------------------------
# Base-fare distribution
# ------------------------------------------------------------

print("\n1. Zero distance by base-fare status")
print("-" * 70)

base_fare_distribution = (
    zero_distance_df["fare_category"]
    .value_counts()
    .rename_axis("fare_category")
    .reset_index(name="count")
)

base_fare_distribution["percentage"] = (
    base_fare_distribution["count"]
    / len(zero_distance_df)
    * 100
)

display(
    base_fare_distribution.style.format({
        "percentage": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# Charge-total distribution
# ------------------------------------------------------------

print("\n2. Zero distance by charge-total status")
print("-" * 70)

charge_distribution = (
    zero_distance_df["charge_category"]
    .value_counts()
    .rename_axis("charge_category")
    .reset_index(name="count")
)

charge_distribution["percentage"] = (
    charge_distribution["count"]
    / len(zero_distance_df)
    * 100
)

display(
    charge_distribution.style.format({
        "percentage": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# Provider distribution for zero distance + nonzero base fare
# ------------------------------------------------------------

print("\n3. Providers: zero distance + nonzero base fare")
print("-" * 70)

zero_distance_nonzero_fare = zero_distance_df[
    zero_distance_df["base_fare"] != 0
].copy()

provider_zero_fare = (
    zero_distance_nonzero_fare
    .groupby("provider_code")
    .size()
    .reset_index(name="count")
)

provider_zero_fare["percentage"] = (
    provider_zero_fare["count"]
    / len(zero_distance_nonzero_fare)
    * 100
)

display(
    provider_zero_fare.style.format({
        "percentage": "{:.2f}%"
    })
)


# ------------------------------------------------------------
# Duration distribution
# ------------------------------------------------------------

print("\n4. Duration profile: zero distance + nonzero base fare")
print("-" * 70)

display(
    zero_distance_nonzero_fare[
        ["duration_seconds", "base_fare", "charge_total"]
    ].describe()
)

ZERO-DISTANCE + NONZERO-FARE AUDIT

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

OVERALL ZERO-DISTANCE RESULTS
Total zero-distance records: 1,478,770

1. Zero distance by base-fare status
----------------------------------------------------------------------


,fare_category,count,percentage
0,Nonzero base fare,1471746,99.53%
1,Zero base fare,7024,0.47%



2. Zero distance by charge-total status
----------------------------------------------------------------------


,charge_category,count,percentage
0,Nonzero charge total,1475973,99.81%
1,Zero charge total,2797,0.19%



3. Providers: zero distance + nonzero base fare
----------------------------------------------------------------------


,provider_code,count,percentage
0,1,140009,9.51%
1,2,1321309,89.78%
2,7,10428,0.71%



4. Duration profile: zero distance + nonzero base fare
----------------------------------------------------------------------


,duration_seconds,base_fare,charge_total
count,1.471746e+06,1.471746e+06,1.471746e+06
mean,9.057022e+02,2.261656e+01,2.896153e+01
std,2.454518e+03,3.388508e+01,3.652837e+01
min,-7.235000e+03,-9.990000e+02,-1.147170e+03
25%,4.800000e+01,4.360000e+00,1.054000e+01
50%,7.840000e+02,1.746000e+01,2.293000e+01
75%,1.306000e+03,3.020000e+01,3.619000e+01
max,5.559090e+05,3.237360e+03,3.240610e+03


In [110]:
# ============================================================
# 1.8.12 EXTREME ZERO-DISTANCE RECORD INVESTIGATION
# ============================================================
#
# Purpose:
#   Investigate extreme zero-distance records before deciding
#   whether they should be removed or retained.
#
# We specifically examine:
#   1. Extremely long zero-distance records
#   2. Negative-duration zero-distance records
#   3. Very large fares
#   4. Whether these anomalies are concentrated in certain providers
#
# IMPORTANT:
#   We are investigating first.
#   We are NOT deleting anything yet.
# ============================================================

ZERO_DISTANCE_EXTREME_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

extreme_zero_distance_records = []

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    df_extreme = pd.read_csv(
        file_path,
        usecols=ZERO_DISTANCE_EXTREME_COLUMNS,
        low_memory=False
    )

    # Parse timestamps
    pickup_dt = pd.to_datetime(
        df_extreme["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df_extreme["dropoff_timestamp"],
        errors="coerce"
    )

    # Calculate trip duration
    df_extreme["duration_seconds"] = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    # Keep only zero-distance records
    zero_distance_mask = (
        df_extreme["distance_miles"] == 0
    )

    df_zero = df_extreme.loc[
        zero_distance_mask
    ].copy()

    if len(df_zero) == 0:
        continue

    # Add source file for traceability
    df_zero["source_file"] = file_path.name

    extreme_zero_distance_records.append(df_zero)


# Combine all months
zero_distance_full = pd.concat(
    extreme_zero_distance_records,
    ignore_index=True
)


print()
print("=" * 70)
print("ZERO-DISTANCE EXTREME RECORD INVESTIGATION")
print("=" * 70)

print(
    f"Total zero-distance records: "
    f"{len(zero_distance_full):,}"
)


# ============================================================
# 1. Negative-duration zero-distance records
# ============================================================

negative_duration_zd = zero_distance_full[
    zero_distance_full["duration_seconds"] < 0
].copy()

print()
print("-" * 70)
print("1. NEGATIVE-DURATION ZERO-DISTANCE RECORDS")
print("-" * 70)

print(
    f"Count: {len(negative_duration_zd):,}"
)

print(
    f"Percentage of zero-distance records: "
    f"{len(negative_duration_zd) / len(zero_distance_full) * 100:.4f}%"
)

if len(negative_duration_zd) > 0:

    print()
    print("By provider:")
    print(
        negative_duration_zd["provider_code"]
        .value_counts()
        .sort_index()
    )

    print()
    print("Duration magnitude:")
    print(
        negative_duration_zd["duration_seconds"]
        .abs()
        .describe()
    )


# ============================================================
# 2. Extremely long zero-distance records
# ============================================================

long_zero_distance = zero_distance_full[
    zero_distance_full["duration_seconds"] > 86400
].copy()

print()
print("-" * 70)
print("2. ZERO-DISTANCE RECORDS LONGER THAN 24 HOURS")
print("-" * 70)

print(
    f"Count: {len(long_zero_distance):,}"
)

print(
    f"Percentage: "
    f"{len(long_zero_distance) / len(zero_distance_full) * 100:.4f}%"
)

if len(long_zero_distance) > 0:

    print()
    print("By provider:")

    print(
        long_zero_distance["provider_code"]
        .value_counts()
        .sort_index()
    )

    print()
    print("Duration statistics:")

    print(
        long_zero_distance["duration_seconds"]
        .describe()
    )


# ============================================================
# 3. Very large base fares
# ============================================================

high_fare_zero_distance = zero_distance_full[
    zero_distance_full["base_fare"] > 500
].copy()

print()
print("-" * 70)
print("3. ZERO-DISTANCE RECORDS WITH BASE FARE > $500")
print("-" * 70)

print(
    f"Count: {len(high_fare_zero_distance):,}"
)

print(
    f"Percentage: "
    f"{len(high_fare_zero_distance) / len(zero_distance_full) * 100:.4f}%"
)

if len(high_fare_zero_distance) > 0:

    print()
    print("By provider:")

    print(
        high_fare_zero_distance["provider_code"]
        .value_counts()
        .sort_index()
    )

    print()
    print("Fare statistics:")

    print(
        high_fare_zero_distance[
            ["base_fare", "charge_total"]
        ].describe()
    )


# ============================================================
# 4. Show the most extreme records
# ============================================================

print()
print("-" * 70)
print("4. TOP 20 LONGEST ZERO-DISTANCE RECORDS")
print("-" * 70)

top_longest = zero_distance_full.sort_values(
    "duration_seconds",
    ascending=False
).head(20)

print(
    top_longest[
        [
            "provider_code",
            "pickup_timestamp",
            "dropoff_timestamp",
            "duration_seconds",
            "rider_count",
            "distance_miles",
            "rate_class_id",
            "offline_record_flag",
            "origin_loc_id",
            "dest_loc_id",
            "fare_settlement_method",
            "base_fare",
            "charge_total",
            "source_file"
        ]
    ].to_string(index=False)
)


# ============================================================
# 5. Show the largest-fare zero-distance records
# ============================================================

print()
print("-" * 70)
print("5. TOP 20 HIGHEST-FARE ZERO-DISTANCE RECORDS")
print("-" * 70)

top_fares = zero_distance_full.sort_values(
    "base_fare",
    ascending=False
).head(20)

print(
    top_fares[
        [
            "provider_code",
            "pickup_timestamp",
            "dropoff_timestamp",
            "duration_seconds",
            "rider_count",
            "distance_miles",
            "rate_class_id",
            "offline_record_flag",
            "origin_loc_id",
            "dest_loc_id",
            "fare_settlement_method",
            "base_fare",
            "charge_total",
            "source_file"
        ]
    ].to_string(index=False)
)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

ZERO-DISTANCE EXTREME RECORD INVESTIGATION
Total zero-distance records: 1,478,770

----------------------------------------------------------------------
1. NEGATIVE-DURATION ZERO-DISTANCE RECORDS
----------------------------------------------------------------------
Count: 12
Percentage of zero-distan

In [112]:
# ============================================================
# 1.8.13 ZERO-DISTANCE SPATIAL CONSISTENCY AUDIT
# ============================================================
#
# Purpose:
#   Determine whether zero-distance records are concentrated
#   in same-zone trips or also occur between different zones.
#
# This is important for:
#   - Data-quality decisions
#   - Fare modelling
#   - Taxi + Zone analysis
#   - OD-flow analysis
#
# We DO NOT remove anything yet.
# ============================================================

ZERO_DISTANCE_SPATIAL_COLUMNS = [
    "provider_code",
    "distance_miles",
    "origin_loc_id",
    "dest_loc_id",
    "rate_class_id",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

zero_distance_spatial_results = []

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    df_spatial = pd.read_csv(
        file_path,
        usecols=ZERO_DISTANCE_SPATIAL_COLUMNS,
        low_memory=False
    )

    # Keep zero-distance records only
    zero_mask = (
        df_spatial["distance_miles"] == 0
    )

    df_zero = df_spatial.loc[zero_mask].copy()

    if len(df_zero) == 0:
        continue

    # Determine whether pickup and destination zones are the same
    df_zero["same_origin_destination"] = (
        df_zero["origin_loc_id"]
        == df_zero["dest_loc_id"]
    )

    zero_distance_spatial_results.append(df_zero)


# Combine all months
zero_distance_spatial_df = pd.concat(
    zero_distance_spatial_results,
    ignore_index=True
)


print()
print("=" * 70)
print("ZERO-DISTANCE SPATIAL CONSISTENCY AUDIT")
print("=" * 70)

print(
    f"Total zero-distance records: "
    f"{len(zero_distance_spatial_df):,}"
)


# ============================================================
# 1. Same-zone vs different-zone
# ============================================================

print()
print("-" * 70)
print("1. SAME ORIGIN/DESTINATION VS DIFFERENT")
print("-" * 70)

spatial_category = (
    zero_distance_spatial_df["same_origin_destination"]
    .map({
        True: "Same origin and destination",
        False: "Different origin and destination"
    })
)

spatial_counts = spatial_category.value_counts()

spatial_percentages = (
    spatial_counts
    / len(zero_distance_spatial_df)
    * 100
)

spatial_summary = pd.DataFrame({
    "count": spatial_counts,
    "percentage": spatial_percentages.round(4)
})

print(spatial_summary)


# ============================================================
# 2. Provider × spatial category
# ============================================================

print()
print("-" * 70)
print("2. PROVIDER × SPATIAL CATEGORY")
print("-" * 70)

provider_spatial = pd.crosstab(
    zero_distance_spatial_df["provider_code"],
    zero_distance_spatial_df["same_origin_destination"]
)

provider_spatial.columns = [
    "Different origin/destination",
    "Same origin/destination"
]

provider_spatial["Total"] = (
    provider_spatial["Different origin/destination"]
    + provider_spatial["Same origin/destination"]
)

provider_spatial["Same_%"] = (
    provider_spatial["Same origin/destination"]
    / provider_spatial["Total"]
    * 100
)

provider_spatial["Different_%"] = (
    provider_spatial["Different origin/destination"]
    / provider_spatial["Total"]
    * 100
)

print(provider_spatial)


# ============================================================
# 3. Rate class × spatial category
# ============================================================

print()
print("-" * 70)
print("3. RATE CLASS × SPATIAL CATEGORY")
print("-" * 70)

rate_spatial = pd.crosstab(
    zero_distance_spatial_df["rate_class_id"],
    zero_distance_spatial_df["same_origin_destination"]
)

rate_spatial.columns = [
    "Different origin/destination",
    "Same origin/destination"
]

rate_spatial["Total"] = (
    rate_spatial["Different origin/destination"]
    + rate_spatial["Same origin/destination"]
)

rate_spatial["Same_%"] = (
    rate_spatial["Same origin/destination"]
    / rate_spatial["Total"]
    * 100
)

print(rate_spatial)


# ============================================================
# 4. Settlement method × spatial category
# ============================================================

print()
print("-" * 70)
print("4. SETTLEMENT METHOD × SPATIAL CATEGORY")
print("-" * 70)

settlement_spatial = pd.crosstab(
    zero_distance_spatial_df["fare_settlement_method"],
    zero_distance_spatial_df["same_origin_destination"]
)

settlement_spatial.columns = [
    "Different origin/destination",
    "Same origin/destination"
]

settlement_spatial["Total"] = (
    settlement_spatial["Different origin/destination"]
    + settlement_spatial["Same origin/destination"]
)

settlement_spatial["Same_%"] = (
    settlement_spatial["Same origin/destination"]
    / settlement_spatial["Total"]
    * 100
)

print(settlement_spatial)


# ============================================================
# 5. Top zero-distance OD pairs
# ============================================================

print()
print("-" * 70)
print("5. TOP 30 ZERO-DISTANCE OD PAIRS")
print("-" * 70)

od_zero_distance = (
    zero_distance_spatial_df
    .groupby(
        ["origin_loc_id", "dest_loc_id"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

od_zero_distance["percentage"] = (
    od_zero_distance["count"]
    / len(zero_distance_spatial_df)
    * 100
)

print(
    od_zero_distance.to_string(index=False)
)


# ============================================================
# 6. Different-zone zero-distance records
# ============================================================

different_zone_zero_distance = (
    zero_distance_spatial_df[
        ~zero_distance_spatial_df[
            "same_origin_destination"
        ]
    ]
)

print()
print("-" * 70)
print("6. DIFFERENT-ZONE + ZERO-DISTANCE RECORDS")
print("-" * 70)

print(
    f"Count: {len(different_zone_zero_distance):,}"
)

print(
    f"Percentage of zero-distance records: "
    f"{len(different_zone_zero_distance) / len(zero_distance_spatial_df) * 100:.4f}%"
)

if len(different_zone_zero_distance) > 0:

    print()
    print("Provider distribution:")

    print(
        different_zone_zero_distance[
            "provider_code"
        ]
        .value_counts()
        .sort_index()
    )

    print()
    print("Top different-zone OD pairs:")

    print(
        different_zone_zero_distance
        .groupby(
            ["origin_loc_id", "dest_loc_id"],
            dropna=False
        )
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(30)
        .to_string(index=False)
    )

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

ZERO-DISTANCE SPATIAL CONSISTENCY AUDIT
Total zero-distance records: 1,478,770

----------------------------------------------------------------------
1. SAME ORIGIN/DESTINATION VS DIFFERENT
----------------------------------------------------------------------
                                    count

In [114]:
# ============================================================
# 1.8.14 RIDER COUNT = 0 AUDIT
# ============================================================
#
# Purpose:
#   Investigate records where rider_count = 0.
#
# We need to determine whether these records are:
#   - legitimate operational records,
#   - cancelled/voided trips,
#   - data-entry issues,
#   - or records that should be excluded from modelling.
#
# IMPORTANT:
#   We are investigating only.
#   No records are deleted at this stage.
# ============================================================

RIDER_ZERO_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

rider_zero_results = []

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    df_rider = pd.read_csv(
        file_path,
        usecols=RIDER_ZERO_COLUMNS,
        low_memory=False
    )

    # Keep rider_count = 0
    zero_rider_mask = (
        df_rider["rider_count"] == 0
    )

    df_zero_rider = df_rider.loc[
        zero_rider_mask
    ].copy()

    if len(df_zero_rider) == 0:
        continue

    # Parse timestamps
    pickup_dt = pd.to_datetime(
        df_zero_rider["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df_zero_rider["dropoff_timestamp"],
        errors="coerce"
    )

    df_zero_rider["duration_seconds"] = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    # Add source file
    df_zero_rider["source_file"] = file_path.name

    rider_zero_results.append(df_zero_rider)


# Combine all months
rider_zero_df = pd.concat(
    rider_zero_results,
    ignore_index=True
)


print()
print("=" * 70)
print("RIDER COUNT = 0 AUDIT")
print("=" * 70)

print(
    f"Total rider_count = 0 records: "
    f"{len(rider_zero_df):,}"
)


# ============================================================
# 1. Overall percentage
# ============================================================

TOTAL_ROWS = 48_601_782

print(
    f"Percentage of entire dataset: "
    f"{len(rider_zero_df) / TOTAL_ROWS * 100:.4f}%"
)


# ============================================================
# 2. Provider distribution
# ============================================================

print()
print("-" * 70)
print("1. RIDER COUNT = 0 BY PROVIDER")
print("-" * 70)

provider_rider_zero = (
    rider_zero_df["provider_code"]
    .value_counts()
    .sort_index()
)

provider_rider_zero_pct = (
    provider_rider_zero
    / len(rider_zero_df)
    * 100
)

print(
    pd.DataFrame({
        "count": provider_rider_zero,
        "percentage": provider_rider_zero_pct.round(4)
    })
)


# ============================================================
# 3. Settlement method
# ============================================================

print()
print("-" * 70)
print("2. RIDER COUNT = 0 BY SETTLEMENT METHOD")
print("-" * 70)

settlement_rider_zero = (
    rider_zero_df["fare_settlement_method"]
    .value_counts(dropna=False)
    .sort_index()
)

settlement_rider_zero_pct = (
    settlement_rider_zero
    / len(rider_zero_df)
    * 100
)

print(
    pd.DataFrame({
        "count": settlement_rider_zero,
        "percentage": settlement_rider_zero_pct.round(4)
    })
)


# ============================================================
# 4. Rate class
# ============================================================

print()
print("-" * 70)
print("3. RIDER COUNT = 0 BY RATE CLASS")
print("-" * 70)

rate_rider_zero = (
    rider_zero_df["rate_class_id"]
    .value_counts(dropna=False)
    .sort_index()
)

rate_rider_zero_pct = (
    rate_rider_zero
    / len(rider_zero_df)
    * 100
)

print(
    pd.DataFrame({
        "count": rate_rider_zero,
        "percentage": rate_rider_zero_pct.round(4)
    })
)


# ============================================================
# 5. Offline flag
# ============================================================

print()
print("-" * 70)
print("4. RIDER COUNT = 0 BY OFFLINE RECORD FLAG")
print("-" * 70)

offline_rider_zero = (
    rider_zero_df["offline_record_flag"]
    .value_counts(dropna=False)
)

offline_rider_zero_pct = (
    offline_rider_zero
    / len(rider_zero_df)
    * 100
)

print(
    pd.DataFrame({
        "count": offline_rider_zero,
        "percentage": offline_rider_zero_pct.round(4)
    })
)


# ============================================================
# 6. Fare characteristics
# ============================================================

print()
print("-" * 70)
print("5. FARE / DISTANCE PROFILE")
print("-" * 70)

print(
    rider_zero_df[
        [
            "distance_miles",
            "duration_seconds",
            "base_fare",
            "charge_total"
        ]
    ].describe()
)


# ============================================================
# 7. Zero riders + zero distance
# ============================================================

zero_rider_zero_distance = rider_zero_df[
    rider_zero_df["distance_miles"] == 0
]

print()
print("-" * 70)
print("6. RIDER COUNT = 0 + ZERO DISTANCE")
print("-" * 70)

print(
    f"Count: {len(zero_rider_zero_distance):,}"
)

print(
    f"Percentage of rider_count = 0 records: "
    f"{len(zero_rider_zero_distance) / len(rider_zero_df) * 100:.4f}%"
)


# ============================================================
# 8. Rider count = 0 + zero fare
# ============================================================

zero_rider_zero_fare = rider_zero_df[
    rider_zero_df["base_fare"] == 0
]

print()
print("-" * 70)
print("7. RIDER COUNT = 0 + ZERO BASE FARE")
print("-" * 70)

print(
    f"Count: {len(zero_rider_zero_fare):,}"
)

print(
    f"Percentage of rider_count = 0 records: "
    f"{len(zero_rider_zero_fare) / len(rider_zero_df) * 100:.4f}%"
)


# ============================================================
# 9. Rider count = 0 + zero duration
# ============================================================

zero_rider_zero_duration = rider_zero_df[
    rider_zero_df["duration_seconds"] == 0
]

print()
print("-" * 70)
print("8. RIDER COUNT = 0 + ZERO DURATION")
print("-" * 70)

print(
    f"Count: {len(zero_rider_zero_duration):,}"
)

print(
    f"Percentage of rider_count = 0 records: "
    f"{len(zero_rider_zero_duration) / len(rider_zero_df) * 100:.4f}%"
)


# ============================================================
# 10. Most common combinations
# ============================================================

print()
print("-" * 70)
print("9. COMMON RIDER=0 COMBINATIONS")
print("-" * 70)

combination_counts = (
    rider_zero_df
    .groupby(
        [
            "provider_code",
            "fare_settlement_method",
            "rate_class_id",
            "offline_record_flag"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

print(
    combination_counts.to_string(index=False)
)


# ============================================================
# 11. Representative records
# ============================================================

print()
print("-" * 70)
print("10. SAMPLE RIDER=0 RECORDS")
print("-" * 70)

print(
    rider_zero_df[
        [
            "provider_code",
            "pickup_timestamp",
            "dropoff_timestamp",
            "duration_seconds",
            "distance_miles",
            "rate_class_id",
            "offline_record_flag",
            "origin_loc_id",
            "dest_loc_id",
            "fare_settlement_method",
            "base_fare",
            "charge_total",
            "source_file"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

RIDER COUNT = 0 AUDIT
Total rider_count = 0 records: 231,578
Percentage of entire dataset: 0.4765%

----------------------------------------------------------------------
1. RIDER COUNT = 0 BY PROVIDER
----------------------------------------------------------------------
                count  percent

In [116]:
# ============================================================
# 1.8.15 UNREALISTIC SPEED AUDIT
# ============================================================
#
# Purpose:
#   Calculate trip speed and investigate unusually high speeds.
#
# Formula:
#
#       speed_mph = distance_miles / duration_hours
#
# We will NOT remove records yet.
#
# Important:
#   - Negative durations are not valid for speed calculation.
#   - Zero-duration records cannot produce a meaningful speed.
#   - Zero-distance records have speed = 0 when duration > 0.
#
# Therefore speed is calculated only where:
#
#       distance > 0
#       duration > 0
#
# ============================================================

SPEED_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "origin_loc_id",
    "dest_loc_id",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

speed_results = []

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    df_speed = pd.read_csv(
        file_path,
        usecols=SPEED_COLUMNS,
        low_memory=False
    )

    # --------------------------------------------------------
    # Parse timestamps
    # --------------------------------------------------------

    pickup_dt = pd.to_datetime(
        df_speed["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df_speed["dropoff_timestamp"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Calculate duration
    # --------------------------------------------------------

    df_speed["duration_seconds"] = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    # --------------------------------------------------------
    # Calculate speed only for physically meaningful records
    # --------------------------------------------------------

    valid_speed_mask = (
        (df_speed["distance_miles"] > 0)
        &
        (df_speed["duration_seconds"] > 0)
    )

    df_valid_speed = df_speed.loc[
        valid_speed_mask
    ].copy()

    if len(df_valid_speed) == 0:
        continue

    # Convert seconds → hours
    duration_hours = (
        df_valid_speed["duration_seconds"] / 3600
    )

    # Speed in miles per hour
    df_valid_speed["speed_mph"] = (
        df_valid_speed["distance_miles"]
        / duration_hours
    )

    # Source file
    df_valid_speed["source_file"] = file_path.name

    speed_results.append(df_valid_speed)


# ============================================================
# Combine all months
# ============================================================

speed_df = pd.concat(
    speed_results,
    ignore_index=True
)


print()
print("=" * 70)
print("UNREALISTIC SPEED AUDIT")
print("=" * 70)

print(
    f"Records eligible for speed calculation: "
    f"{len(speed_df):,}"
)


# ============================================================
# 1. Overall speed distribution
# ============================================================

print()
print("-" * 70)
print("1. OVERALL SPEED DISTRIBUTION")
print("-" * 70)

print(
    speed_df["speed_mph"].describe(
        percentiles=[
            .50,
            .75,
            .90,
            .95,
            .99,
            .995,
            .999
        ]
    )
)


# ============================================================
# 2. High-speed thresholds
# ============================================================

print()
print("-" * 70)
print("2. HIGH-SPEED THRESHOLD COUNTS")
print("-" * 70)

speed_thresholds = [
    40,
    50,
    60,
    70,
    80,
    90,
    100,
    120,
    150
]

speed_threshold_results = []

for threshold in speed_thresholds:

    count = (
        speed_df["speed_mph"] > threshold
    ).sum()

    percentage = (
        count
        / len(speed_df)
        * 100
    )

    speed_threshold_results.append({
        "threshold_mph": threshold,
        "count": count,
        "percentage": percentage
    })

speed_threshold_table = pd.DataFrame(
    speed_threshold_results
)

print(
    speed_threshold_table.to_string(
        index=False
    )
)


# ============================================================
# 3. Extremely high speeds
# ============================================================

extreme_speed = speed_df[
    speed_df["speed_mph"] > 100
].copy()

print()
print("-" * 70)
print("3. RECORDS WITH SPEED > 100 MPH")
print("-" * 70)

print(
    f"Count: {len(extreme_speed):,}"
)

print(
    f"Percentage of speed-valid records: "
    f"{len(extreme_speed) / len(speed_df) * 100:.4f}%"
)

if len(extreme_speed) > 0:

    print()
    print("By provider:")

    print(
        extreme_speed[
            "provider_code"
        ]
        .value_counts()
        .sort_index()
    )


# ============================================================
# 4. Extreme speed statistics
# ============================================================

print()
print("-" * 70)
print("4. SPEED > 100 MPH PROFILE")
print("-" * 70)

if len(extreme_speed) > 0:

    print(
        extreme_speed[
            [
                "speed_mph",
                "distance_miles",
                "duration_seconds",
                "base_fare",
                "charge_total"
            ]
        ].describe()
    )


# ============================================================
# 5. Top 30 fastest records
# ============================================================

print()
print("-" * 70)
print("5. TOP 30 FASTEST TRIPS")
print("-" * 70)

top_fastest = (
    speed_df
    .sort_values(
        "speed_mph",
        ascending=False
    )
    .head(30)
)

print(
    top_fastest[
        [
            "provider_code",
            "pickup_timestamp",
            "dropoff_timestamp",
            "speed_mph",
            "distance_miles",
            "duration_seconds",
            "rider_count",
            "rate_class_id",
            "origin_loc_id",
            "dest_loc_id",
            "fare_settlement_method",
            "base_fare",
            "charge_total",
            "source_file"
        ]
    ].to_string(index=False)
)


# ============================================================
# 6. Provider-level speed statistics
# ============================================================

print()
print("-" * 70)
print("6. SPEED DISTRIBUTION BY PROVIDER")
print("-" * 70)

provider_speed = (
    speed_df
    .groupby("provider_code")["speed_mph"]
    .describe(
        percentiles=[
            .50,
            .90,
            .95,
            .99,
            .999
        ]
    )
)

print(provider_speed)


# ============================================================
# 7. Rate-class speed statistics
# ============================================================

print()
print("-" * 70)
print("7. SPEED DISTRIBUTION BY RATE CLASS")
print("-" * 70)

rate_speed = (
    speed_df
    .groupby("rate_class_id")["speed_mph"]
    .describe(
        percentiles=[
            .50,
            .90,
            .95,
            .99,
            .999
        ]
    )
)

print(rate_speed)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv

UNREALISTIC SPEED AUDIT
Records eligible for speed calculation: 46,488,457

----------------------------------------------------------------------
1. OVERALL SPEED DISTRIBUTION
----------------------------------------------------------------------
count    4.648846e+07
mean     2.507372e+01
std      3.

MemoryError: Unable to allocate 1.04 GiB for an array with shape (3, 46488457) and data type float64

In [118]:
# ============================================================
# 1.8.15A MEMORY-EFFICIENT EXTREME SPEED INVESTIGATION
# ============================================================
#
# Purpose:
#   Find the fastest trips WITHOUT constructing/sorting a
#   massive 46M-row DataFrame.
#
# We keep only the top 50 records across all files.
#
# This is much more memory-efficient than:
#
#     speed_df.sort_values(...)
#
# ============================================================

import heapq
import pandas as pd


SPEED_TOP_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "origin_loc_id",
    "dest_loc_id",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

# ------------------------------------------------------------
# Keep only the top 50 records.
#
# Each heap item:
#     (speed, record)
#
# ------------------------------------------------------------

TOP_N = 50

top_speed_heap = []


for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    df = pd.read_csv(
        file_path,
        usecols=SPEED_TOP_COLUMNS,
        low_memory=False
    )

    # Parse timestamps
    pickup_dt = pd.to_datetime(
        df["pickup_timestamp"],
        errors="coerce"
    )

    dropoff_dt = pd.to_datetime(
        df["dropoff_timestamp"],
        errors="coerce"
    )

    # Duration
    duration_seconds = (
        dropoff_dt - pickup_dt
    ).dt.total_seconds()

    # Valid speed records
    valid_mask = (
        (df["distance_miles"] > 0)
        &
        (duration_seconds > 0)
    )

    valid = df.loc[valid_mask].copy()

    if len(valid) == 0:
        continue

    valid["duration_seconds"] = (
        duration_seconds.loc[valid_mask]
    )

    valid["speed_mph"] = (
        valid["distance_miles"]
        /
        (valid["duration_seconds"] / 3600)
    )

    valid["source_file"] = file_path.name

    # --------------------------------------------------------
    # Keep only a small candidate set from this file.
    # --------------------------------------------------------

    candidate = valid.nlargest(
        TOP_N,
        "speed_mph"
    )

    for _, row in candidate.iterrows():

        speed = float(row["speed_mph"])

        record = row.to_dict()

        if len(top_speed_heap) < TOP_N:

            heapq.heappush(
                top_speed_heap,
                (speed, record)
            )

        elif speed > top_speed_heap[0][0]:

            heapq.heapreplace(
                top_speed_heap,
                (speed, record)
            )


# ============================================================
# Convert final top records into a DataFrame
# ============================================================

top_speed_records = [
    record
    for speed, record in top_speed_heap
]

top_speed_df = pd.DataFrame(
    top_speed_records
).sort_values(
    "speed_mph",
    ascending=False
)


print()
print("=" * 70)
print("TOP 50 FASTEST TRIPS")
print("=" * 70)

print(
    top_speed_df[
        [
            "provider_code",
            "pickup_timestamp",
            "dropoff_timestamp",
            "speed_mph",
            "distance_miles",
            "duration_seconds",
            "rider_count",
            "rate_class_id",
            "origin_loc_id",
            "dest_loc_id",
            "fare_settlement_method",
            "base_fare",
            "charge_total",
            "source_file"
        ]
    ].to_string(index=False)
)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv


ParserError: Error tokenizing data. C error: out of memory

In [121]:
# Release large objects from the previous speed analysis
for var in [
    "df",
    "speed_df",
    "extreme_speed",
    "top_fastest",
    "global_top_fastest"
]:
    if var in globals():
        del globals()[var]

import gc
gc.collect()

print("Memory cleanup completed.")

Memory cleanup completed.


In [123]:
# ============================================================
# 1.8.15A - Memory-Efficient Extreme Speed Investigation
# ============================================================
#
# Purpose:
#   Find the fastest trips without loading an entire monthly
#   CSV into memory.
#
# Why chunking?
#   Each monthly file is ~300-470 MB on disk.
#   Pandas expands CSV data substantially in RAM.
#   Reading in chunks keeps memory usage manageable.
#
# We keep only the top 50 fastest records from each file.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import heapq
import gc

SPEED_TOP_COLUMNS = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "distance_miles",
    "rider_count",
    "rate_class_id",
    "origin_loc_id",
    "dest_loc_id",
    "fare_settlement_method",
    "base_fare",
    "charge_total"
]

TOP_N_PER_FILE = 50
CHUNK_SIZE = 100_000

global_top_fastest = []

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"\nProcessing: {file_path.name}")

    file_top = []

    for chunk in pd.read_csv(
        file_path,
        usecols=SPEED_TOP_COLUMNS,
        chunksize=CHUNK_SIZE,
        low_memory=True
    ):

        # ----------------------------------------------------
        # Parse timestamps
        # ----------------------------------------------------
        pickup_dt = pd.to_datetime(
            chunk["pickup_timestamp"],
            errors="coerce"
        )

        dropoff_dt = pd.to_datetime(
            chunk["dropoff_timestamp"],
            errors="coerce"
        )

        # ----------------------------------------------------
        # Calculate duration
        # ----------------------------------------------------
        duration_seconds = (
            dropoff_dt - pickup_dt
        ).dt.total_seconds()

        # ----------------------------------------------------
        # Calculate speed only for valid records
        # ----------------------------------------------------
        valid = (
            duration_seconds > 0
            & chunk["distance_miles"].notna()
            & (chunk["distance_miles"] > 0)
        )

        if not valid.any():
            continue

        temp = chunk.loc[valid].copy()

        temp["duration_seconds"] = duration_seconds.loc[valid]

        temp["speed_mph"] = (
            temp["distance_miles"]
            / (temp["duration_seconds"] / 3600)
        )

        # ----------------------------------------------------
        # Keep only fastest records from this chunk
        # ----------------------------------------------------
        temp = temp.nlargest(
            TOP_N_PER_FILE,
            "speed_mph"
        )

        file_top.append(temp)

        del temp
        gc.collect()

    # --------------------------------------------------------
    # Combine the small top-N results from this file
    # --------------------------------------------------------
    if file_top:

        file_top = pd.concat(
            file_top,
            ignore_index=True
        )

        file_top = file_top.nlargest(
            TOP_N_PER_FILE,
            "speed_mph"
        )

        # Add source file
        file_top["source_file"] = file_path.name

        global_top_fastest.append(file_top)

        print(
            f"  Top speed in file: "
            f"{file_top['speed_mph'].max():,.2f} mph"
        )

    del file_top
    gc.collect()


# ============================================================
# Combine results from all 12 files
# ============================================================

global_top_fastest = pd.concat(
    global_top_fastest,
    ignore_index=True
)

global_top_fastest = global_top_fastest.nlargest(
    100,
    "speed_mph"
).reset_index(drop=True)


# ============================================================
# Display the fastest records
# ============================================================

print("\n" + "=" * 80)
print("TOP 100 FASTEST VALID TRIPS")
print("=" * 80)

display(
    global_top_fastest[
        [
            "source_file",
            "provider_code",
            "pickup_timestamp",
            "dropoff_timestamp",
            "duration_seconds",
            "distance_miles",
            "speed_mph",
            "rider_count",
            "rate_class_id",
            "origin_loc_id",
            "dest_loc_id",
            "fare_settlement_method",
            "base_fare",
            "charge_total"
        ]
    ].head(50)
)


Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
  Top speed in file: 2,332,021.50 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
  Top speed in file: 3,119,835.60 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
  Top speed in file: 1,722,263.40 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
  Top speed in file: 1,875,959.76 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
  Top speed in file: 2,740,823.80 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
  Top speed in file: 4,034,270.10 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
  Top speed in file: 2,439,476.85 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
  Top speed in file: 1,897,379.00 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
  Top speed in file: 3,748,310.40 mph

Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
  Top speed in file: 2,218,899.96 mph

Processing: Urban_Flow_Analyt

,source_file,provider_code,pickup_timestamp,dropoff_timestamp,duration_seconds,distance_miles,speed_mph,rider_count,rate_class_id,origin_loc_id,dest_loc_id,fare_settlement_method,base_fare,charge_total
0,Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv,2,2026-02-20 08:00:00,2026-02-20 08:03:00,180.0,240830.74,4.816615e+06,NaN,NaN,151,166,0,15.07,20.57
1,Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv,2,2025-09-04 01:02:00,2025-09-04 01:06:00,240.0,268951.34,4.034270e+06,NaN,NaN,114,114,0,5.10,9.85
2,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,2,2025-12-05 06:53:00,2025-12-05 06:55:00,120.0,124943.68,3.748310e+06,NaN,NaN,42,41,0,8.32,9.82
3,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,2,2025-05-05 20:12:00,2025-05-05 20:14:00,120.0,103994.52,3.119836e+06,NaN,NaN,33,33,0,49.46,50.96
4,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,2,2025-12-21 06:30:00,2025-12-21 06:37:00,420.0,322576.17,2.764939e+06,NaN,NaN,82,260,0,6.34,9.84
5,Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,2,2025-08-29 00:28:00,2025-08-29 00:34:00,360.0,274082.38,2.740824e+06,NaN,NaN,114,249,0,6.13,10.88
6,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,2,2025-05-23 07:48:00,2025-05-23 07:51:00,180.0,122104.37,2.442087e+06,NaN,NaN,159,159,0,-11.18,-9.68
7,Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,2,2025-10-04 02:22:00,2025-10-04 02:26:00,240.0,162631.79,2.439477e+06,NaN,NaN,79,107,0,4.69,9.44
8,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,2,2025-04-17 20:27:00,2025-04-17 20:29:00,120.0,77734.05,2.332022e+06,NaN,NaN,79,79,0,-4.75,2.40
9,Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv,2,2026-01-07 17:51:00,2026-01-07 17:56:00,300.0,184908.33,2.218900e+06,NaN,NaN,164,186,0,12.18,16.93


In [127]:
# ============================================================
# 1.8.15B - Memory-Efficient Speed Threshold Summary
# ============================================================
#
# Purpose:
#   Recalculate speed anomaly counts without storing all
#   46+ million speed records in memory.
#
# This is safe for the large dataset because processing is
# performed in chunks.
# ============================================================

import pandas as pd
import numpy as np
import gc

SPEED_COLUMNS = [
    "pickup_timestamp",
    "dropoff_timestamp",
    "distance_miles"
]

SPEED_THRESHOLDS = [40, 50, 60, 70, 80, 90, 100, 120, 150]

threshold_counts = {threshold: 0 for threshold in SPEED_THRESHOLDS}

eligible_count = 0

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    file_eligible = 0
    file_counts = {threshold: 0 for threshold in SPEED_THRESHOLDS}

    for chunk in pd.read_csv(
        file_path,
        usecols=SPEED_COLUMNS,
        chunksize=100_000,
        low_memory=True
    ):

        pickup_dt = pd.to_datetime(
            chunk["pickup_timestamp"],
            errors="coerce"
        )

        dropoff_dt = pd.to_datetime(
            chunk["dropoff_timestamp"],
            errors="coerce"
        )

        duration_seconds = (
            dropoff_dt - pickup_dt
        ).dt.total_seconds()

        valid = (
            duration_seconds > 0
            & chunk["distance_miles"].notna()
            & (chunk["distance_miles"] > 0)
        )

        if not valid.any():
            continue

        speeds = (
            chunk.loc[valid, "distance_miles"]
            / (duration_seconds.loc[valid] / 3600)
        )

        file_eligible += len(speeds)

        for threshold in SPEED_THRESHOLDS:
            file_counts[threshold] += (
                speeds > threshold
            ).sum()

        del speeds

    eligible_count += file_eligible

    for threshold in SPEED_THRESHOLDS:
        threshold_counts[threshold] += file_counts[threshold]

    print(f"  Eligible trips: {file_eligible:,}")

    gc.collect()


# ============================================================
# Create final summary
# ============================================================

speed_summary_df = pd.DataFrame([
    {
        "speed_threshold_mph": threshold,
        "anomaly_count": threshold_counts[threshold],
        "percentage_of_eligible_trips":
            threshold_counts[threshold]
            / eligible_count
            * 100
    }
    for threshold in SPEED_THRESHOLDS
])


print("\n" + "=" * 80)
print("SPEED ANOMALY SUMMARY")
print("=" * 80)

print(f"Eligible trips: {eligible_count:,}")
print(f"Total dataset: 48,601,782")

display(speed_summary_df)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
  Eligible trips: 3,935,784
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
  Eligible trips: 4,527,573
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
  Eligible trips: 4,254,377
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
  Eligible trips: 3,842,899
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
  Eligible trips: 3,526,258
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
  Eligible trips: 4,193,836
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
  Eligible trips: 4,360,731
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
  Eligible trips: 4,119,324
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
  Eligible trips: 4,246,985
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
  Eligible trips: 3,679,819
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
  Eligible trips: 3,359,268
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.

,speed_threshold_mph,anomaly_count,percentage_of_eligible_trips
0,40,170458,0.355490
1,50,30298,0.063186
2,60,17238,0.035950
3,70,15014,0.031312
4,80,13589,0.028340
5,90,12555,0.026183
6,100,11899,0.024815
7,120,10678,0.022269
8,150,9589,0.019998


In [129]:
# ============================================================
# 1.9 - MASTER DATA QUALITY DECISION TABLE
# ============================================================
#
# This table documents:
#   1. What anomaly was detected
#   2. How it was detected
#   3. How many records were affected
#   4. Percentage of total dataset affected
#   5. Treatment
#   6. Reason for treatment
#
# IMPORTANT:
#   Raw data will NOT be modified.
#   These rules will be applied later when creating
#   analytical/modeling datasets.
# ============================================================

TOTAL_ROWS = 48_601_782

master_quality = pd.DataFrame([
    
    {
        "anomaly": "Negative trip duration",
        "detection_rule": "dropoff_timestamp < pickup_timestamp",
        "count": 1_942,
        "percentage_of_total": 1_942 / TOTAL_ROWS * 100,
        "treatment": "Exclude from ETA modeling; retain raw record and flag",
        "reason": "Negative elapsed time is invalid as an ETA target"
    },

    {
        "anomaly": "Zero trip duration",
        "detection_rule": "trip_duration_seconds == 0",
        "count": 649_668,
        "percentage_of_total": 649_668 / TOTAL_ROWS * 100,
        "treatment": "Exclude from ETA modeling; retain for other analyses where appropriate",
        "reason": "A zero-duration trip cannot provide a valid elapsed-time target"
    },

    {
        "anomaly": "Provider 7 zero-duration records",
        "detection_rule": "provider_code == 7 AND trip_duration_seconds == 0",
        "count": 642_536,
        "percentage_of_total": 642_536 / TOTAL_ROWS * 100,
        "treatment": "Exclude from ETA modeling; retain for fare and spatial analysis",
        "reason": "Provider 7 systematically records zero duration despite meaningful distance and fare"
    },

    {
        "anomaly": "Zero distance",
        "detection_rule": "distance_miles == 0",
        "count": 1_478_770,
        "percentage_of_total": 1_478_770 / TOTAL_ROWS * 100,
        "treatment": "Retain and flag",
        "reason": "Zero distance occurs frequently and may still contain useful fare, time and location information"
    },

    {
        "anomaly": "Zero rider count",
        "detection_rule": "rider_count == 0",
        "count": 231_578,
        "percentage_of_total": 231_578 / TOTAL_ROWS * 100,
        "treatment": "Retain and flag",
        "reason": "Records may otherwise represent valid trips; no evidence justifies blanket deletion or imputation"
    },

    {
        "anomaly": "Unrealistic speed",
        "detection_rule": "calculated speed > 100 mph",
        "count": 11_899,
        "percentage_of_total": 11_899 / TOTAL_ROWS * 100,
        "treatment": "Flag; exclude from ETA modeling",
        "reason": "Extreme calculated speeds are inconsistent with realistic taxi travel and indicate distance/time anomalies"
    }
])

display(master_quality)

,anomaly,detection_rule,count,percentage_of_total,treatment,reason
0,Negative trip duration,dropoff_timestamp < pickup_timestamp,1942,0.003996,Exclude from ETA modeling; retain raw record a...,Negative elapsed time is invalid as an ETA target
1,Zero trip duration,trip_duration_seconds == 0,649668,1.336716,Exclude from ETA modeling; retain for other an...,A zero-duration trip cannot provide a valid el...
2,Provider 7 zero-duration records,provider_code == 7 AND trip_duration_seconds == 0,642536,1.322042,Exclude from ETA modeling; retain for fare and...,Provider 7 systematically records zero duratio...
3,Zero distance,distance_miles == 0,1478770,3.042625,Retain and flag,Zero distance occurs frequently and may still ...
4,Zero rider count,rider_count == 0,231578,0.476480,Retain and flag,Records may otherwise represent valid trips; n...
5,Unrealistic speed,calculated speed > 100 mph,11899,0.024483,Flag; exclude from ETA modeling,Extreme calculated speeds are inconsistent wit...


In [131]:
# ============================================================
# 1.9.1 - Save Master Data Quality Table
# ============================================================

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "data_quality_decision_table.csv"

master_quality.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to:")
print(OUTPUT_PATH)

Saved to:
C:\Users\arudk\Downloads\UrbanFlow_AI\outputs\data_quality_decision_table.csv


## 1.10.1 Monetary Columns

The taxi dataset contains several monetary fields representing the
fare and its associated charges.

The main monetary variables considered in this investigation are:

- `base_fare`
- `surcharge_misc`
- `transit_tax`
- `driver_tip_payment`
- `toll_total`
- `service_improvement_fee`
- `charge_total`
- `zone_congestion_fee`
- `Airport_fee`
- `congestion_relief_fee`

The purpose of this section is to determine whether negative or
extreme monetary values represent invalid observations, legitimate
adjustments, or special settlement records.

No records will be removed at this stage.

In [134]:
# ============================================================
# 1.10.1 - Define Monetary Columns
# ============================================================

MONETARY_COLUMNS = [
    "base_fare",
    "surcharge_misc",
    "transit_tax",
    "driver_tip_payment",
    "toll_total",
    "service_improvement_fee",
    "charge_total",
    "zone_congestion_fee",
    "Airport_fee",
    "congestion_relief_fee"
]

print("Monetary columns:")
for column in MONETARY_COLUMNS:
    print(f"  - {column}")

Monetary columns:
  - base_fare
  - surcharge_misc
  - transit_tax
  - driver_tip_payment
  - toll_total
  - service_improvement_fee
  - charge_total
  - zone_congestion_fee
  - Airport_fee
  - congestion_relief_fee


## 1.10.2 Negative Monetary Value Audit

For each monetary variable, we calculate:

- number of negative values
- percentage of the complete dataset affected
- number of zero values
- number of positive values
- minimum value
- maximum value

This provides an initial overview of the monetary data quality.

The analysis is performed in chunks because the complete taxi dataset
contains approximately 48.6 million records.

In [137]:
# ============================================================
# 1.10.2 - Full Dataset Monetary Audit
# ============================================================

MONETARY_AUDIT_COLUMNS = [
    "base_fare",
    "surcharge_misc",
    "transit_tax",
    "driver_tip_payment",
    "toll_total",
    "service_improvement_fee",
    "charge_total",
    "zone_congestion_fee",
    "Airport_fee",
    "congestion_relief_fee"
]

# ------------------------------------------------------------
# Initialize counters
# ------------------------------------------------------------

monetary_stats = {
    column: {
        "negative": 0,
        "zero": 0,
        "positive": 0,
        "missing": 0,
        "minimum": np.inf,
        "maximum": -np.inf
    }
    for column in MONETARY_AUDIT_COLUMNS
}


# ------------------------------------------------------------
# Process each monthly file
# ------------------------------------------------------------

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    for chunk in pd.read_csv(
        file_path,
        usecols=MONETARY_AUDIT_COLUMNS,
        chunksize=100_000,
        low_memory=True
    ):

        for column in MONETARY_AUDIT_COLUMNS:

            values = chunk[column]

            # Count missing
            monetary_stats[column]["missing"] += values.isna().sum()

            # Count negative
            monetary_stats[column]["negative"] += (
                values < 0
            ).sum()

            # Count zero
            monetary_stats[column]["zero"] += (
                values == 0
            ).sum()

            # Count positive
            monetary_stats[column]["positive"] += (
                values > 0
            ).sum()

            # Update minimum / maximum
            if values.notna().any():

                current_min = values.min()
                current_max = values.max()

                monetary_stats[column]["minimum"] = min(
                    monetary_stats[column]["minimum"],
                    current_min
                )

                monetary_stats[column]["maximum"] = max(
                    monetary_stats[column]["maximum"],
                    current_max
                )

        del chunk

    gc.collect()


# ------------------------------------------------------------
# Convert results to DataFrame
# ------------------------------------------------------------

monetary_audit_df = pd.DataFrame.from_dict(
    monetary_stats,
    orient="index"
).reset_index()

monetary_audit_df = monetary_audit_df.rename(
    columns={"index": "column"}
)

monetary_audit_df["negative_percentage"] = (
    monetary_audit_df["negative"]
    / TOTAL_ROWS
    * 100
)

monetary_audit_df["zero_percentage"] = (
    monetary_audit_df["zero"]
    / TOTAL_ROWS
    * 100
)

monetary_audit_df["positive_percentage"] = (
    monetary_audit_df["positive"]
    / TOTAL_ROWS
    * 100
)

monetary_audit_df["missing_percentage"] = (
    monetary_audit_df["missing"]
    / TOTAL_ROWS
    * 100
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    monetary_audit_df[
        [
            "column",
            "negative",
            "negative_percentage",
            "zero",
            "zero_percentage",
            "positive",
            "positive_percentage",
            "missing",
            "missing_percentage",
            "minimum",
            "maximum"
        ]
    ]
)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


,column,negative,negative_percentage,zero,zero_percentage,positive,positive_percentage,missing,missing_percentage,minimum,maximum
0,base_fare,2400031,4.938154,24577,0.050568,46177174,95.011278,0,0.000000,-2555.20,325478.05
1,surcharge_misc,366095,0.753254,28034908,57.682881,20200779,41.563865,0,0.000000,-17.39,133.60
2,transit_tax,670316,1.379200,683914,1.407179,47247552,97.213621,0,0.000000,-21.74,5243.38
3,driver_tip_payment,1481,0.003047,19727503,40.590082,28872798,59.406871,0,0.000000,-333.33,960.94
4,toll_total,65691,0.135162,45215051,93.031673,3321040,6.833165,0,0.000000,-148.17,1400.00
5,service_improvement_fee,709250,1.459309,1035624,2.130835,46856908,96.409856,0,0.000000,-1.00,2.50
6,charge_total,875399,1.801166,6699,0.013783,47719684,98.185050,0,0.000000,-2560.20,325528.45
7,zone_congestion_fee,555805,1.143590,3660390,7.531391,31980319,65.800713,12405268,25.524307,-2.50,2.50
8,Airport_fee,152252,0.313264,32858197,67.606980,3186065,6.555449,12405268,25.524307,-2.00,27.00
9,congestion_relief_fee,475447,0.978250,13282073,27.328366,34844262,71.693384,0,0.000000,-0.75,1.75


## 1.10.3 Negative Monetary Values by Settlement Method

A negative monetary value does not necessarily indicate a corrupted
trip. It may be associated with a particular settlement or
transaction type.

The dataset contains the following `fare_settlement_method` values:

- 0 = Flex Fare
- 1 = Credit card
- 2 = Cash
- 3 = No charge
- 4 = Dispute
- 5 = Unknown
- 6 = Voided trip

We therefore examine negative monetary records in relation to the
settlement method.

This is important because a concentration of negative values within
specific settlement categories may indicate adjustments, disputes,
voids, or other special transactions rather than ordinary trip fares.

No records are removed in this step.

In [140]:
# ============================================================
# 1.10.3 - Negative Monetary Values by Settlement Method
# ============================================================
#
# Purpose:
#   Determine whether negative monetary values are concentrated
#   within particular fare settlement methods.
#
# Memory strategy:
#   Process the large dataset in chunks.
# ============================================================

SETTLEMENT_COLUMNS = [
    "fare_settlement_method",
    "base_fare",
    "surcharge_misc",
    "transit_tax",
    "driver_tip_payment",
    "toll_total",
    "service_improvement_fee",
    "charge_total",
    "zone_congestion_fee",
    "Airport_fee",
    "congestion_relief_fee"
]

negative_settlement_counts = {
    column: {}
    for column in MONETARY_COLUMNS
}

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    for chunk in pd.read_csv(
        file_path,
        usecols=SETTLEMENT_COLUMNS,
        chunksize=100_000,
        low_memory=True
    ):

        for monetary_column in MONETARY_COLUMNS:

            negative_mask = (
                chunk[monetary_column] < 0
            )

            if not negative_mask.any():
                continue

            negative_rows = chunk.loc[
                negative_mask,
                ["fare_settlement_method"]
            ]

            counts = (
                negative_rows[
                    "fare_settlement_method"
                ]
                .value_counts(dropna=False)
            )

            for settlement_method, count in counts.items():

                negative_settlement_counts[
                    monetary_column
                ][settlement_method] = (
                    negative_settlement_counts[
                        monetary_column
                    ].get(settlement_method, 0)
                    + int(count)
                )

        del chunk

    gc.collect()


# ------------------------------------------------------------
# Convert results into a readable table
# ------------------------------------------------------------

settlement_rows = []

for monetary_column, settlement_counts in (
    negative_settlement_counts.items()
):

    total_negative = sum(
        settlement_counts.values()
    )

    for settlement_method, count in (
        settlement_counts.items()
    ):

        settlement_rows.append({
            "monetary_column": monetary_column,
            "fare_settlement_method": settlement_method,
            "negative_count": count,
            "percentage_of_negative_values": (
                count / total_negative * 100
                if total_negative > 0
                else 0
            )
        })

negative_settlement_df = pd.DataFrame(
    settlement_rows
)

display(
    negative_settlement_df.sort_values(
        [
            "monetary_column",
            "negative_count"
        ],
        ascending=[True, False]
    )
)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


,monetary_column,fare_settlement_method,negative_count,percentage_of_negative_values
38,Airport_fee,4,96678,63.498673
39,Airport_fee,2,36648,24.070620
40,Airport_fee,3,18857,12.385387
41,Airport_fee,1,69,0.045320
4,base_fare,0,1694004,70.582588
0,base_fare,4,455956,18.997921
1,base_fare,2,162267,6.761038
2,base_fare,3,86005,3.583495
3,base_fare,1,1799,0.074957
29,charge_total,4,456046,52.095787


## 1.10.4 Negative Base Fare by Settlement Method

`base_fare` is the primary target for the fare prediction model.

Therefore, negative `base_fare` values require special attention.

We examine:

- number of negative base-fare records
- percentage of negative base-fare records
- percentage of each settlement method that is negative

The second measure is particularly useful because a settlement method
may contain many negative records simply because it is common in the
dataset.

In [143]:
# ============================================================
# 1.10.4 - Negative Base Fare by Settlement Method
# ============================================================

BASE_FARE_SETTLEMENT_COLUMNS = [
    "fare_settlement_method",
    "base_fare"
]

settlement_basefare_stats = {}

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    for chunk in pd.read_csv(
        file_path,
        usecols=BASE_FARE_SETTLEMENT_COLUMNS,
        chunksize=100_000,
        low_memory=True
    ):

        # ----------------------------------------------------
        # Count all records by settlement method
        # ----------------------------------------------------

        total_counts = (
            chunk["fare_settlement_method"]
            .value_counts(dropna=False)
        )

        # ----------------------------------------------------
        # Count negative base fares
        # ----------------------------------------------------

        negative_counts = (
            chunk.loc[
                chunk["base_fare"] < 0,
                "fare_settlement_method"
            ]
            .value_counts(dropna=False)
        )

        # ----------------------------------------------------
        # Accumulate results
        # ----------------------------------------------------

        for settlement_method, count in total_counts.items():

            if settlement_method not in settlement_basefare_stats:
                settlement_basefare_stats[
                    settlement_method
                ] = {
                    "total": 0,
                    "negative": 0
                }

            settlement_basefare_stats[
                settlement_method
            ]["total"] += int(count)

        for settlement_method, count in negative_counts.items():

            if settlement_method not in settlement_basefare_stats:
                settlement_basefare_stats[
                    settlement_method
                ] = {
                    "total": 0,
                    "negative": 0
                }

            settlement_basefare_stats[
                settlement_method
            ]["negative"] += int(count)

        del chunk

    gc.collect()


# ------------------------------------------------------------
# Build result table
# ------------------------------------------------------------

basefare_settlement_rows = []

for settlement_method, stats in (
    settlement_basefare_stats.items()
):

    total = stats["total"]
    negative = stats["negative"]

    basefare_settlement_rows.append({
        "fare_settlement_method": settlement_method,
        "total_records": total,
        "negative_base_fare": negative,
        "negative_percentage_of_method": (
            negative / total * 100
            if total > 0
            else 0
        ),
        "percentage_of_all_negative_base_fares": (
            negative / 2_400_031 * 100
            if 2_400_031 > 0
            else 0
        )
    })

basefare_settlement_df = pd.DataFrame(
    basefare_settlement_rows
)

display(
    basefare_settlement_df.sort_values(
        "negative_base_fare",
        ascending=False
    )
)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


,fare_settlement_method,total_records,negative_base_fare,negative_percentage_of_method,percentage_of_all_negative_base_fares
4,0,12405268,1694004,13.655521,70.582588
2,4,979752,455956,46.537899,18.997921
1,2,4459788,162267,3.638446,6.761038
3,3,277468,86005,30.996367,3.583495
0,1,30479504,1799,0.005902,0.074957
5,5,2,0,0.000000,0.000000


## 1.10.5 Negative Base Fare: Trip Context

Negative `base_fare` records are compared with their surrounding trip
characteristics.

The objective is to determine whether these records resemble normal
taxi trips or whether they form a distinct operational/settlement
population.

We do not assume that a negative fare is erroneous simply because
the value is below zero.

The analysis therefore focuses on the characteristics associated
with these records.

In [146]:
# ============================================================
# 1.10.5 - Context of Negative Base Fare Records
# ============================================================

CONTEXT_COLUMNS = [
    "provider_code",
    "fare_settlement_method",
    "rate_class_id",
    "rider_count",
    "distance_miles",
    "pickup_timestamp",
    "dropoff_timestamp",
    "base_fare",
    "charge_total"
]

negative_basefare_context = []

# We intentionally collect only a sample of negative records
# from each file so that memory remains small.

SAMPLE_PER_FILE = 2_000

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    file_samples = []

    for chunk in pd.read_csv(
        file_path,
        usecols=CONTEXT_COLUMNS,
        chunksize=100_000,
        low_memory=True
    ):

        negative_rows = chunk.loc[
            chunk["base_fare"] < 0
        ]

        if len(negative_rows) > 0:

            # Keep a limited sample from each chunk
            sample = negative_rows.sample(
                n=min(
                    SAMPLE_PER_FILE,
                    len(negative_rows)
                ),
                random_state=42
            )

            file_samples.append(sample)

        del chunk

    if file_samples:

        file_sample = pd.concat(
            file_samples,
            ignore_index=True
        )

        # Keep the file-level sample manageable
        file_sample = file_sample.head(
            SAMPLE_PER_FILE
        )

        file_sample["source_file"] = (
            file_path.name
        )

        negative_basefare_context.append(
            file_sample
        )

    gc.collect()


negative_basefare_context_df = pd.concat(
    negative_basefare_context,
    ignore_index=True
)

print(
    f"Collected context sample: "
    f"{len(negative_basefare_context_df):,} records"
)

display(
    negative_basefare_context_df.head(50)
)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv
Collected context sample: 24,000 records


,provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,fare_settlement_method,base_fare,charge_total,source_file
0,2,2025-04-01 19:51:14,2025-04-01 19:53:50,3.0,0.26,1.0,3,-4.4,-11.65,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
1,2,2025-04-01 13:14:51,2025-04-01 13:22:14,1.0,1.04,1.0,3,-8.6,-13.35,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
2,2,2025-04-01 03:17:45,2025-04-01 03:45:15,1.0,26.74,4.0,4,-125.5,-148.50,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
3,2,2025-04-01 10:36:49,2025-04-01 10:40:32,2.0,0.27,2.0,4,-70.0,-80.19,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
4,2,2025-04-01 12:33:45,2025-04-01 12:48:29,1.0,1.09,1.0,4,-13.5,-17.50,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
5,2,2025-04-01 21:20:20,2025-04-01 21:27:25,1.0,0.87,1.0,3,-8.6,-14.35,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
6,2,2025-04-01 08:57:34,2025-04-01 09:07:47,1.0,0.84,1.0,4,-10.0,-14.75,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
7,2,2025-04-01 19:55:09,2025-04-01 20:02:08,2.0,1.24,1.0,2,-8.6,-15.85,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
8,2,2025-04-01 18:33:15,2025-04-01 18:42:59,1.0,0.87,1.0,4,-10.0,-17.25,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
9,2,2025-04-01 14:19:37,2025-04-01 14:46:37,4.0,14.47,1.0,4,-54.1,-69.29,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv


## 1.10.6 Negative Base Fare by Settlement Method: Interpretation

The negative base-fare audit shows that negative values are highly
concentrated in specific fare settlement methods.

In particular, negative base fares are uncommon within Flex Fare
records but substantially more frequent among Cash, No Charge, and
Dispute records.

This concentration indicates that negative monetary values may be
associated with special settlement or transaction conditions rather
than representing ordinary completed taxi fares.

Therefore, negative `base_fare` values will not be blindly replaced
with zero, imputed, or deleted at this stage.

Further investigation is required before defining the training
population for the fare prediction model.

The raw observations will be retained for traceability.

## 1.10.7 Negative vs Non-Negative Base Fare Profile

To determine whether negative base-fare records resemble ordinary
taxi trips, the dataset is divided into two groups:

- Negative base fare: `base_fare < 0`
- Non-negative base fare: `base_fare >= 0`

For each group, we examine:

- number of records
- distance
- trip duration
- rider count
- charge total

The comparison helps determine whether negative fares should be
treated as invalid targets for fare prediction or retained as part
of the normal fare population.

The analysis is performed in chunks to avoid loading the complete
dataset into memory.

In [150]:
# ============================================================
# 1.10.7 - Negative vs Non-Negative Base Fare Profile
# ============================================================

PROFILE_COLUMNS = [
    "base_fare",
    "charge_total",
    "distance_miles",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count"
]

# ------------------------------------------------------------
# Initialize group statistics
# ------------------------------------------------------------

fare_groups = {
    "negative_base_fare": {
        "count": 0,
        "distance_sum": 0.0,
        "duration_sum": 0.0,
        "rider_sum": 0.0,
        "charge_sum": 0.0,
        "distance_count": 0,
        "duration_count": 0,
        "rider_count": 0,
        "charge_count": 0
    },

    "non_negative_base_fare": {
        "count": 0,
        "distance_sum": 0.0,
        "duration_sum": 0.0,
        "rider_sum": 0.0,
        "charge_sum": 0.0,
        "distance_count": 0,
        "duration_count": 0,
        "rider_count": 0,
        "charge_count": 0
    }
}


# ------------------------------------------------------------
# Process files in chunks
# ------------------------------------------------------------

for file_path in sorted(TAXI_DIR.glob("*.csv")):

    print(f"Processing: {file_path.name}")

    for chunk in pd.read_csv(
        file_path,
        usecols=PROFILE_COLUMNS,
        chunksize=100_000,
        low_memory=True
    ):

        # ----------------------------------------------------
        # Calculate trip duration
        # ----------------------------------------------------

        pickup_dt = pd.to_datetime(
            chunk["pickup_timestamp"],
            errors="coerce"
        )

        dropoff_dt = pd.to_datetime(
            chunk["dropoff_timestamp"],
            errors="coerce"
        )

        duration_seconds = (
            dropoff_dt - pickup_dt
        ).dt.total_seconds()

        # ----------------------------------------------------
        # Define groups
        # ----------------------------------------------------

        groups = {
            "negative_base_fare": chunk["base_fare"] < 0,
            "non_negative_base_fare": chunk["base_fare"] >= 0
        }

        # ----------------------------------------------------
        # Aggregate each group
        # ----------------------------------------------------

        for group_name, mask in groups.items():

            group = chunk.loc[mask]

            fare_groups[group_name]["count"] += len(group)

            # Distance
            distance = group["distance_miles"].dropna()

            fare_groups[group_name]["distance_sum"] += (
                distance.sum()
            )

            fare_groups[group_name]["distance_count"] += (
                distance.count()
            )

            # Duration
            duration = duration_seconds.loc[mask].dropna()

            fare_groups[group_name]["duration_sum"] += (
                duration.sum()
            )

            fare_groups[group_name]["duration_count"] += (
                duration.count()
            )

            # Rider count
            riders = group["rider_count"].dropna()

            fare_groups[group_name]["rider_sum"] += (
                riders.sum()
            )

            fare_groups[group_name]["rider_count"] += (
                riders.count()
            )

            # Charge total
            charge = group["charge_total"].dropna()

            fare_groups[group_name]["charge_sum"] += (
                charge.sum()
            )

            fare_groups[group_name]["charge_count"] += (
                charge.count()
            )

        del chunk

    gc.collect()


# ------------------------------------------------------------
# Build summary table
# ------------------------------------------------------------

fare_profile_rows = []

for group_name, stats in fare_groups.items():

    fare_profile_rows.append({
        "group": group_name,
        "record_count": stats["count"],

        "mean_distance_miles": (
            stats["distance_sum"]
            / stats["distance_count"]
        ),

        "mean_duration_seconds": (
            stats["duration_sum"]
            / stats["duration_count"]
        ),

        "mean_rider_count": (
            stats["rider_sum"]
            / stats["rider_count"]
        ),

        "mean_charge_total": (
            stats["charge_sum"]
            / stats["charge_count"]
        )
    })


fare_profile_df = pd.DataFrame(
    fare_profile_rows
)

display(fare_profile_df)

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


,group,record_count,mean_distance_miles,mean_duration_seconds,mean_rider_count,mean_charge_total
0,negative_base_fare,2400031,13.579385,1091.663736,1.390924,-6.552736
1,non_negative_base_fare,46201751,6.504831,1066.714994,1.282532,29.639309


## 1.10.8 Verify Fare Settlement Method Codes

Before interpreting monetary anomalies by settlement method, we verify the
actual settlement codes present in the raw dataset.

This is important because the observed frequencies should be consistent
with the data dictionary. We will not assume that a numeric code represents
a particular settlement category until this is verified.

In [153]:
# ============================================================
# 1.10.8 Verify Fare Settlement Method Codes
# ============================================================

# We already know the dataset is very large.
# Therefore, scan only the required column instead of loading
# the entire dataset into memory.

settlement_counts = {}

for file_path in taxi_files:
    print(f"Processing: {file_path.name}")

    # Read only the settlement column
    for chunk in pd.read_csv(
        file_path,
        usecols=["fare_settlement_method"],
        chunksize=500_000
    ):
        counts = chunk["fare_settlement_method"].value_counts(dropna=False)

        for code, count in counts.items():
            settlement_counts[code] = settlement_counts.get(code, 0) + count


# Convert the result into a clean table
settlement_code_table = (
    pd.Series(settlement_counts, name="record_count")
    .rename_axis("fare_settlement_method")
    .reset_index()
    .sort_values("fare_settlement_method")
    .reset_index(drop=True)
)

# Calculate percentage of the complete dataset
settlement_code_table["percentage_of_all_records"] = (
    settlement_code_table["record_count"] / TOTAL_ROWS * 100
)

settlement_code_table

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


,fare_settlement_method,record_count,percentage_of_all_records
0,0,12405268,25.524307
1,1,30479504,62.712729
2,2,4459788,9.176182
3,3,277468,0.570901
4,4,979752,2.015877
5,5,2,0.000004


## 1.10.9 Monetary Anomaly Decision Analysis

The previous analysis identified negative values in several monetary
variables.

We now investigate whether negative `base_fare` records are associated
with particular settlement codes.

This analysis is used to determine whether negative fares should be
retained, flagged, or excluded from the fare prediction modelling dataset.

We will use the actual settlement codes observed in the raw data and will
not assume descriptive labels unless they are confirmed by the source
documentation.

In [156]:
# ============================================================
# 1.10.9 Negative Base Fare by Actual Settlement Code
# ============================================================

negative_base_by_settlement = {}

total_by_settlement = {}

for file_path in taxi_files:
    print(f"Processing: {file_path.name}")

    # Read only the columns needed for this investigation
    for chunk in pd.read_csv(
        file_path,
        usecols=[
            "fare_settlement_method",
            "base_fare"
        ],
        chunksize=500_000
    ):

        # Total records by settlement code
        total_counts = (
            chunk["fare_settlement_method"]
            .value_counts(dropna=False)
        )

        # Negative base-fare records by settlement code
        negative_counts = (
            chunk.loc[
                chunk["base_fare"] < 0,
                "fare_settlement_method"
            ]
            .value_counts(dropna=False)
        )

        # Accumulate totals
        for code, count in total_counts.items():
            total_by_settlement[code] = (
                total_by_settlement.get(code, 0) + count
            )

        # Accumulate negative counts
        for code, count in negative_counts.items():
            negative_base_by_settlement[code] = (
                negative_base_by_settlement.get(code, 0) + count
            )


# Create final comparison table
settlement_basefare_table = pd.DataFrame({
    "fare_settlement_method": sorted(total_by_settlement.keys()),
    "total_records": [
        total_by_settlement[code]
        for code in sorted(total_by_settlement.keys())
    ],
    "negative_base_fare": [
        negative_base_by_settlement.get(code, 0)
        for code in sorted(total_by_settlement.keys())
    ]
})

# Percentage of each settlement-code group that has negative base fare
settlement_basefare_table["negative_percentage_of_code"] = (
    settlement_basefare_table["negative_base_fare"]
    / settlement_basefare_table["total_records"]
    * 100
)

# Percentage contribution to all negative base-fare records
settlement_basefare_table["percentage_of_all_negative_base_fare"] = (
    settlement_basefare_table["negative_base_fare"]
    / 2_400_031
    * 100
)

settlement_basefare_table

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


,fare_settlement_method,total_records,negative_base_fare,negative_percentage_of_code,percentage_of_all_negative_base_fare
0,0,12405268,1694004,13.655521,70.582588
1,1,30479504,1799,0.005902,0.074957
2,2,4459788,162267,3.638446,6.761038
3,3,277468,86005,30.996367,3.583495
4,4,979752,455956,46.537899,18.997921
5,5,2,0,0.000000,0.000000


## 1.10.10 Investigate Negative Charge Totals

`charge_total` represents the overall billed amount for the trip.

We therefore investigate negative `charge_total` values separately from
negative individual fare components.

The purpose is to determine whether negative base fares consistently result
in negative final charges or whether some records contain positive overall
charges despite having negative fare components.

This distinction will help us decide how monetary anomalies should be
handled for the fare prediction model.

In [159]:
# ============================================================
# 1.10.10 Negative Charge Total Analysis
# ============================================================

# Counters for the complete dataset
negative_charge_total = 0
zero_charge_total = 0
positive_charge_total = 0

# Relationship between base_fare and charge_total
negative_base_negative_charge = 0
negative_base_positive_charge = 0
negative_base_zero_charge = 0

positive_base_negative_charge = 0

for file_path in taxi_files:
    print(f"Processing: {file_path.name}")

    for chunk in pd.read_csv(
        file_path,
        usecols=["base_fare", "charge_total"],
        chunksize=500_000
    ):

        base = chunk["base_fare"]
        charge = chunk["charge_total"]

        # ----------------------------------------------------
        # Overall charge_total distribution
        # ----------------------------------------------------

        negative_charge_total += (charge < 0).sum()
        zero_charge_total += (charge == 0).sum()
        positive_charge_total += (charge > 0).sum()

        # ----------------------------------------------------
        # Relationship between base_fare and charge_total
        # ----------------------------------------------------

        negative_base = base < 0
        non_negative_base = base >= 0

        negative_base_negative_charge += (
            negative_base & (charge < 0)
        ).sum()

        negative_base_positive_charge += (
            negative_base & (charge > 0)
        ).sum()

        negative_base_zero_charge += (
            negative_base & (charge == 0)
        ).sum()

        positive_base_negative_charge += (
            non_negative_base & (charge < 0)
        ).sum()


# ------------------------------------------------------------
# Create summary table
# ------------------------------------------------------------

charge_summary = pd.DataFrame({
    "category": [
        "Negative charge_total",
        "Zero charge_total",
        "Positive charge_total",
        "Negative base_fare + negative charge_total",
        "Negative base_fare + zero charge_total",
        "Negative base_fare + positive charge_total",
        "Non-negative base_fare + negative charge_total"
    ],
    "record_count": [
        negative_charge_total,
        zero_charge_total,
        positive_charge_total,
        negative_base_negative_charge,
        negative_base_zero_charge,
        negative_base_positive_charge,
        positive_base_negative_charge
    ]
})

charge_summary["percentage_of_all_records"] = (
    charge_summary["record_count"] / TOTAL_ROWS * 100
)

charge_summary

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


,category,record_count,percentage_of_all_records
0,Negative charge_total,875399,1.801166
1,Zero charge_total,6699,0.013783
2,Positive charge_total,47719684,98.185050
3,Negative base_fare + negative charge_total,869827,1.789702
4,Negative base_fare + zero charge_total,0,0.000000
5,Negative base_fare + positive charge_total,1530204,3.148452
6,Non-negative base_fare + negative charge_total,5572,0.011465


## 1.10.11 Check Internal Fare Consistency

The dataset contains several monetary components in addition to
`charge_total`.

We investigate whether these components approximately reconcile with
the recorded `charge_total`.

We do not assume an exact accounting formula because the data dictionary
does not explicitly define `charge_total` as the mathematical sum of all
monetary columns.

Instead, we calculate the difference and investigate how large the
discrepancies are.

This helps identify potentially corrupted monetary records without
incorrectly removing legitimate financial adjustments.

In [177]:
# ============================================================
# 1.10.11 Internal Fare Consistency Check
# ============================================================

monetary_columns = [
    "base_fare",
    "surcharge_misc",
    "transit_tax",
    "driver_tip_payment",
    "toll_total",
    "service_improvement_fee",
    "zone_congestion_fee",
    "Airport_fee",
    "congestion_relief_fee"
]

# We will calculate the difference between the sum of the
# monetary components and the recorded charge_total.

difference_count = 0
difference_abs_sum = 0
difference_abs_max = 0

# We keep a sample of differences so that we do not have to
# store tens of millions of values in memory.
difference_values = []

for file_path in taxi_files:
    print(f"Processing: {file_path.name}")

    for chunk in pd.read_csv(
        file_path,
        usecols=monetary_columns + ["charge_total"],
        chunksize=500_000
    ):

        # Add the available monetary components for each trip
        component_sum = chunk[monetary_columns].sum(axis=1)

        # Compare calculated component total with recorded charge
        difference = component_sum - chunk["charge_total"]

        # Absolute difference
        abs_difference = difference.abs()

        difference_count += len(difference)
        difference_abs_sum += abs_difference.sum()

        # Track the largest difference found
        difference_abs_max = max(
            difference_abs_max,
            abs_difference.max()
        )

        # Keep a manageable random sample
        difference_values.extend(
            abs_difference.sample(
                min(10_000, len(abs_difference)),
                random_state=42
            ).tolist()
        )


# Convert sampled differences to a Series
difference_sample = pd.Series(difference_values)


# Create summary table
fare_consistency_summary = pd.DataFrame({
    "metric": [
        "Records checked",
        "Mean absolute difference (sample)",
        "Median absolute difference (sample)",
        "95th percentile absolute difference (sample)",
        "99th percentile absolute difference (sample)",
        "Maximum absolute difference"
    ],
    "value": [
        difference_count,
        difference_sample.mean(),
        difference_sample.median(),
        difference_sample.quantile(0.95),
        difference_sample.quantile(0.99),
        difference_abs_max
    ]
})

fare_consistency_summary

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


,metric,value
0,Records checked,4.860178e+07
1,Mean absolute difference (sample),1.312455e+00
2,Median absolute difference (sample),0.000000e+00
3,95th percentile absolute difference (sample),4.800000e+00
4,99th percentile absolute difference (sample),9.690000e+00
5,Maximum absolute difference,4.397900e+02


## 1.11 Final Data Quality Summary

This section consolidates the major data-quality findings identified during
the audit of the complete taxi dataset.

The original records are preserved. Instead of applying blanket deletion
rules, anomaly flags and model-specific filtering rules are used.

This approach allows us to maintain traceability while preventing known
data-quality problems from contaminating models where they are unsuitable.

In [180]:
# ============================================================
# 1.11 Final Data Quality Summary
# ============================================================

data_quality_summary = pd.DataFrame({
    "anomaly_type": [
        "Negative trip duration",
        "Zero trip duration",
        "Provider 7 zero-duration records",
        "Zero distance",
        "Zero rider count",
        "Unrealistic speed >100 mph",
        "Negative base fare",
        "Negative charge total"
    ],

    "affected_records": [
        1_942,
        649_668,
        642_536,
        1_478_770,
        231_578,
        11_899,
        2_400_031,
        875_399
    ],

    "percentage_of_total": [
        1_942 / TOTAL_ROWS * 100,
        649_668 / TOTAL_ROWS * 100,
        642_536 / TOTAL_ROWS * 100,
        1_478_770 / TOTAL_ROWS * 100,
        231_578 / TOTAL_ROWS * 100,
        11_899 / TOTAL_ROWS * 100,
        2_400_031 / TOTAL_ROWS * 100,
        875_399 / TOTAL_ROWS * 100
    ],

    "decision": [
        "Exclude from ETA; retain raw",
        "Exclude from ETA; retain raw",
        "Exclude from ETA; retain for fare/spatial analysis",
        "Retain and flag",
        "Retain and flag",
        "Exclude from ETA; retain raw",
        "Retain and flag; investigate for fare modelling",
        "Retain and flag"
    ]
})

data_quality_summary

,anomaly_type,affected_records,percentage_of_total,decision
0,Negative trip duration,1942,0.003996,Exclude from ETA; retain raw
1,Zero trip duration,649668,1.336716,Exclude from ETA; retain raw
2,Provider 7 zero-duration records,642536,1.322042,Exclude from ETA; retain for fare/spatial anal...
3,Zero distance,1478770,3.042625,Retain and flag
4,Zero rider count,231578,0.476480,Retain and flag
5,Unrealistic speed >100 mph,11899,0.024483,Exclude from ETA; retain raw
6,Negative base fare,2400031,4.938154,Retain and flag; investigate for fare modelling
7,Negative charge total,875399,1.801166,Retain and flag


In [182]:
# Save the final summary for use in the report
output_path = PROJECT_ROOT / "outputs" / "data_quality_summary.csv"

data_quality_summary.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: C:\Users\arudk\Downloads\UrbanFlow_AI\outputs\data_quality_summary.csv


## 1.12 Define Model-Specific Data Quality Rules

The audit showed that different anomalies affect different analytical tasks.

Therefore, we retain the original taxi records and create explicit quality
flags. Model-specific datasets will then apply only the filtering rules
required for that particular task.

This prevents valid information from being unnecessarily discarded while
ensuring that clearly invalid target values do not enter the corresponding
models.

In [185]:
# ============================================================
# 1.12 Model-Specific Data Quality Rules
# ============================================================

# These rules will be applied later when the modelling datasets
# are created.

QUALITY_RULES = {
    "eta": {
        "exclude_negative_duration": True,
        "exclude_zero_duration": True,
        "exclude_provider_7": True,
        "exclude_unrealistic_speed": True,
        "speed_threshold_mph": 100
    },

    "fare": {
        "exclude_negative_target": True,
        "retain_zero_distance": True,
        "retain_zero_rider_count": True
    },

    "zone_od": {
        "retain_zero_distance": True,
        "retain_zero_rider_count": True,
        "retain_negative_fare_records": True
    }
}

QUALITY_RULES

{'eta': {'exclude_negative_duration': True,
  'exclude_zero_duration': True,
  'exclude_provider_7': True,
  'exclude_unrealistic_speed': True,
  'speed_threshold_mph': 100},
 'fare': {'exclude_negative_target': True,
  'retain_zero_distance': True,
  'retain_zero_rider_count': True},
 'zone_od': {'retain_zero_distance': True,
  'retain_zero_rider_count': True,
  'retain_negative_fare_records': True}}

## 1.13 Data Quality Flags

Quality flags are created instead of overwriting the original variables.

The flags provide traceability and allow different models to apply
different filtering rules.

The original values are preserved.

In [188]:
# ============================================================
# 1.13 Data Quality Flag Function
# ============================================================

def add_quality_flags(df):
    """
    Add data-quality flags to a taxi DataFrame.

    The original columns are not modified.
    """

    # --------------------------------------------------------
    # Trip duration
    # --------------------------------------------------------

    df["trip_duration_seconds"] = (
        pd.to_datetime(df["dropoff_timestamp"])
        - pd.to_datetime(df["pickup_timestamp"])
    ).dt.total_seconds()

    df["negative_duration_flag"] = (
        df["trip_duration_seconds"] < 0
    )

    df["zero_duration_flag"] = (
        df["trip_duration_seconds"] == 0
    )

    # --------------------------------------------------------
    # Distance
    # --------------------------------------------------------

    df["zero_distance_flag"] = (
        df["distance_miles"] == 0
    )

    # --------------------------------------------------------
    # Rider count
    # --------------------------------------------------------

    df["zero_rider_count_flag"] = (
        df["rider_count"] == 0
    )

    # --------------------------------------------------------
    # Monetary anomalies
    # --------------------------------------------------------

    df["negative_base_fare_flag"] = (
        df["base_fare"] < 0
    )

    df["negative_charge_total_flag"] = (
        df["charge_total"] < 0
    )

    # --------------------------------------------------------
    # Speed
    # --------------------------------------------------------

    # Only calculate speed when duration and distance are
    # both positive. This avoids division by zero and avoids
    # creating meaningless speed values.

    df["speed_mph"] = np.nan

    valid_speed = (
        (df["trip_duration_seconds"] > 0)
        & (df["distance_miles"] > 0)
    )

    df.loc[valid_speed, "speed_mph"] = (
        df.loc[valid_speed, "distance_miles"]
        / (df.loc[valid_speed, "trip_duration_seconds"] / 3600)
    )

    df["unrealistic_speed_flag"] = (
        df["speed_mph"] > 100
    )

    # --------------------------------------------------------
    # Provider 7 zero-duration condition
    # --------------------------------------------------------

    df["provider_7_zero_duration_flag"] = (
        (df["provider_code"] == 7)
        & (df["trip_duration_seconds"] == 0)
    )

    return df

In [190]:
# ============================================================
# 1.13.1 Test Quality Flags on a Small Sample
# ============================================================

test_sample = pd.read_csv(
    taxi_files[0],
    nrows=10_000
)

test_sample = add_quality_flags(test_sample)

# Display the newly created quality columns
quality_flag_columns = [
    "trip_duration_seconds",
    "negative_duration_flag",
    "zero_duration_flag",
    "zero_distance_flag",
    "zero_rider_count_flag",
    "negative_base_fare_flag",
    "negative_charge_total_flag",
    "speed_mph",
    "unrealistic_speed_flag",
    "provider_7_zero_duration_flag"
]

test_sample[quality_flag_columns].head(10)

,trip_duration_seconds,negative_duration_flag,zero_duration_flag,zero_distance_flag,zero_rider_count_flag,negative_base_fare_flag,negative_charge_total_flag,speed_mph,unrealistic_speed_flag,provider_7_zero_duration_flag
0,1579.0,False,False,False,False,False,False,21.659278,False,False
1,644.0,False,False,False,False,False,False,21.074534,False,False
2,665.0,False,False,False,False,False,False,29.287218,False,False
3,259.0,False,False,False,False,False,False,8.339768,False,False
4,962.0,False,False,False,False,False,False,1.609148,False,False
5,0.0,False,True,False,False,False,False,NaN,False,True
6,1173.0,False,False,False,False,False,False,27.437340,False,False
7,1015.0,False,False,False,False,False,False,31.176355,False,False
8,90.0,False,False,False,False,False,False,20.000000,False,False
9,1779.0,False,False,False,False,False,False,33.632378,False,False


## 1.14.1 Create the Taxi Data Processing Function

The raw taxi dataset contains more than 48 million records.

To make the dataset practical to work with, the raw files are processed
in chunks rather than loaded into memory at once.

The original raw files remain unchanged.

The processed data will contain the variables required for data analysis,
fare modelling, ETA modelling, and taxi-zone analysis.

In [193]:
# ============================================================
# 1.14.1 Taxi Data Processing Function
# ============================================================

def process_taxi_chunk(df):
    """
    Process one chunk of the raw taxi dataset.

    The original raw variables are preserved.
    Additional derived variables and quality flags are added.
    """

    # --------------------------------------------------------
    # Convert timestamps
    # --------------------------------------------------------

    df["pickup_timestamp"] = pd.to_datetime(
        df["pickup_timestamp"],
        errors="coerce"
    )

    df["dropoff_timestamp"] = pd.to_datetime(
        df["dropoff_timestamp"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Trip duration
    # --------------------------------------------------------

    df["trip_duration_seconds"] = (
        df["dropoff_timestamp"]
        - df["pickup_timestamp"]
    ).dt.total_seconds()

    # --------------------------------------------------------
    # Quality flags
    # --------------------------------------------------------

    df["negative_duration_flag"] = (
        df["trip_duration_seconds"] < 0
    )

    df["zero_duration_flag"] = (
        df["trip_duration_seconds"] == 0
    )

    df["zero_distance_flag"] = (
        df["distance_miles"] == 0
    )

    df["zero_rider_count_flag"] = (
        df["rider_count"] == 0
    )

    df["negative_base_fare_flag"] = (
        df["base_fare"] < 0
    )

    df["negative_charge_total_flag"] = (
        df["charge_total"] < 0
    )

    # --------------------------------------------------------
    # Speed
    # --------------------------------------------------------

    valid_speed = (
        (df["trip_duration_seconds"] > 0)
        & (df["distance_miles"] > 0)
    )

    df["speed_mph"] = np.nan

    df.loc[valid_speed, "speed_mph"] = (
        df.loc[valid_speed, "distance_miles"]
        / (df.loc[valid_speed, "trip_duration_seconds"] / 3600)
    )

    df["unrealistic_speed_flag"] = (
        df["speed_mph"] > 100
    )

    # --------------------------------------------------------
    # Provider 7 zero-duration flag
    # --------------------------------------------------------

    df["provider_7_zero_duration_flag"] = (
        (df["provider_code"] == 7)
        & (df["trip_duration_seconds"] == 0)
    )

    # --------------------------------------------------------
    # Calendar features
    # --------------------------------------------------------

    df["pickup_date"] = df["pickup_timestamp"].dt.date
    df["pickup_hour"] = df["pickup_timestamp"].dt.hour
    df["pickup_day_of_week"] = df["pickup_timestamp"].dt.dayofweek
    df["pickup_month"] = df["pickup_timestamp"].dt.month

    # Monday = 0, Sunday = 6
    df["is_weekend"] = (
        df["pickup_day_of_week"] >= 5
    ).astype("int8")

    return df

In [195]:
# ============================================================
# 1.14.2 Test Processing Function
# ============================================================

test_processing = pd.read_csv(
    taxi_files[0],
    nrows=10_000
)

test_processing = process_taxi_chunk(
    test_processing
)

new_columns = [
    "trip_duration_seconds",
    "negative_duration_flag",
    "zero_duration_flag",
    "zero_distance_flag",
    "zero_rider_count_flag",
    "negative_base_fare_flag",
    "negative_charge_total_flag",
    "speed_mph",
    "unrealistic_speed_flag",
    "provider_7_zero_duration_flag",
    "pickup_date",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "is_weekend"
]

test_processing[new_columns].head(10)

,trip_duration_seconds,negative_duration_flag,zero_duration_flag,zero_distance_flag,zero_rider_count_flag,negative_base_fare_flag,negative_charge_total_flag,speed_mph,unrealistic_speed_flag,provider_7_zero_duration_flag,pickup_date,pickup_hour,pickup_day_of_week,pickup_month,is_weekend
0,1579.0,False,False,False,False,False,False,21.659278,False,False,2025-04-01,0,1,4,0
1,644.0,False,False,False,False,False,False,21.074534,False,False,2025-04-01,0,1,4,0
2,665.0,False,False,False,False,False,False,29.287218,False,False,2025-04-01,0,1,4,0
3,259.0,False,False,False,False,False,False,8.339768,False,False,2025-04-01,0,1,4,0
4,962.0,False,False,False,False,False,False,1.609148,False,False,2025-04-01,0,1,4,0
5,0.0,False,True,False,False,False,False,NaN,False,True,2025-04-01,0,1,4,0
6,1173.0,False,False,False,False,False,False,27.437340,False,False,2025-04-01,0,1,4,0
7,1015.0,False,False,False,False,False,False,31.176355,False,False,2025-04-01,0,1,4,0
8,90.0,False,False,False,False,False,False,20.000000,False,False,2025-04-01,0,1,4,0
9,1779.0,False,False,False,False,False,False,33.632378,False,False,2025-04-01,0,1,4,0


## 1.14 Process Full Taxi Dataset

The 12 raw monthly taxi files contain approximately 48.6 million records.

The data is processed month by month in chunks to avoid memory problems.

The raw CSV files are never modified.

Each processed monthly file contains the original taxi variables together
with the quality flags and basic time features required for subsequent
fare, ETA, demand, hotspot, and OD analysis.

In [198]:
# ============================================================
# 1.14 Full Taxi Data Processing
# ============================================================

PROCESSED_TAXI_DIR = (
    PROJECT_ROOT / "data" / "processed" / "taxi_clean"
)

PROCESSED_TAXI_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for file_path in taxi_files:

    print("=" * 70)
    print(f"Processing: {file_path.name}")

    # Output filename
    output_file = (
        PROCESSED_TAXI_DIR
        / f"{file_path.stem}_processed.parquet"
    )

    processed_chunks = []

    # Read the monthly file in manageable pieces
    for chunk in pd.read_csv(
        file_path,
        chunksize=500_000
    ):

        chunk = process_taxi_chunk(chunk)

        processed_chunks.append(chunk)

    # Combine the chunks for this month
    monthly_df = pd.concat(
        processed_chunks,
        ignore_index=True
    )

    # Save as compressed Parquet
    monthly_df.to_parquet(
        output_file,
        index=False,
        compression="snappy"
    )

    print(f"Rows processed: {len(monthly_df):,}")
    print(f"Saved: {output_file}")

    # Free memory before processing next month
    del monthly_df
    del processed_chunks

Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_13152\2062814718.py:28: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Rows processed: 3,970,553
Saved: C:\Users\arudk\Downloads\UrbanFlow_AI\data\processed\taxi_clean\Urban_Flow_Analytics_Taxi_Dataset_2025-04_processed.parquet
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_13152\2062814718.py:28: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Rows processed: 4,591,845
Saved: C:\Users\arudk\Downloads\UrbanFlow_AI\data\processed\taxi_clean\Urban_Flow_Analytics_Taxi_Dataset_2025-05_processed.parquet
Processing: Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv


...

Rows processed: 3,724,889
Saved: C:\Users\arudk\Downloads\UrbanFlow_AI\data\processed\taxi_clean\Urban_Flow_Analytics_Taxi_Dataset_2026-01_processed.parquet
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_13152\2062814718.py:28: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Rows processed: 3,399,866
Saved: C:\Users\arudk\Downloads\UrbanFlow_AI\data\processed\taxi_clean\Urban_Flow_Analytics_Taxi_Dataset_2026-02_processed.parquet
Processing: Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv


C:\Users\arudk\AppData\Local\Temp\ipykernel_13152\2062814718.py:28: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(


Rows processed: 3,952,451
Saved: C:\Users\arudk\Downloads\UrbanFlow_AI\data\processed\taxi_clean\Urban_Flow_Analytics_Taxi_Dataset_2026-03_processed.parquet
